# **Model Fitting**

### **$$\boldsymbol{J}_{CLRi} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertible(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_J(X_c, X_r, Y_c, Y_r, V, beta_c, beta_r, K):
    J = 0
    for k in range(K):
        V_k = V[k]
        J_c = ((Y_c - X_c @ beta_c[k]).T @ V_k @ (Y_c - X_c @ beta_c[k]))
        J_r = ((Y_r - X_r @ beta_r[k]).T @ V_k @ (Y_r - X_r @ beta_r[k]))
        J += J_c + J_r
    return J

def model_fitting(X_c, X_r, Y_c, Y_r, V, beta_c, beta_r, K, epsilon):
    test = 1
    t =

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_c.shape[0])])

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertible(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertible(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):

            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2).sum()
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                test = 1
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1

            clusters[i] = new_cluster

        J = compute_J(X_c, X_r, Y_c, Y_r, V, new_beta_c, new_beta_r, K)

        beta_c, beta_r = new_beta_c, new_beta_r

    return beta_c, beta_r, clusters, V

def run_multiple_experiments(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, r, epsilon):
    best_J = float('inf')
    best_J_reg = None
    best_J_cl = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, clusters, V_final = model_fitting(
            X_c, X_r, Y_c, Y_r, V, beta_c, beta_r, K, epsilon)

        J_final = compute_J(X_c, X_r, Y_c, Y_r, V_final, beta_c_final, beta_r_final, K)

        if J_final < best_J:
            best_J = J_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertible(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_J2a(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p = X_i0.shape[1]

    for k in range(K):
        V_k = V[k]
        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum((X_i0 - g_i[k])**2, axis=1)
        dist_s = np.sum((X_s0 - g_s[k])**2, axis=1)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroids(X_i0, X_s0, V, K):
    n_samples = X_i0.shape[0]
    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s

def model_fitting2a(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):
    """Ajuste do modelo."""
    test = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))


    J, J_reg, J_cl = compute_J2a(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0)


    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_c.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertible(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertible(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroids(X_i0, X_s0, V, K)

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):

            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2) +
                alpha * np.sum([(X_i0[i, j] - g_i[k, j])**2 + (X_s0[i, j] - g_s[k, j])**2 for j in range(X_i0.shape[1])])
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                test = 1
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1

            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_J2a(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, new_beta_c, new_beta_r, K, alpha, g_i, g_s, X_i0, X_s0)

        cluster_positions = {k: [] for k in range(K)}
        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)
        beta_c, beta_r = new_beta_c, new_beta_r

    return beta_c, beta_r, g_i, g_s, clusters, V

def run_multiple_experiments_V(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):
    best_J = float('inf')
    best_J_reg = None
    best_J_cl = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, clusters, V_final = model_fitting2a(
            X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_J2a(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:])

        if J_final < best_J:
            best_J = J_final
            best_J_reg = J_reg_final
            best_J_cl = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED-LC1} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertibleV2(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_JV2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p = X_i0.shape[1]

    for k in range(K):
        V_k = V[k]

        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum([(X_i0[:, j] - g_i[k, j])**2 * lambda_kj[k, j] for j in range(p)], axis=0)
        dist_s = np.sum([(X_s0[:, j] - g_s[k, j])**2 * lambda_kj[k, j] for j in range(p)], axis=0)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroidsV2(X_i0, X_s0, V, K):
    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s

def update_lambda_kjV2(X_i0, X_s0, V, g_i, g_s, K, lambda_kj_prev, epsilon):
    p = X_i0.shape[1]
    lambda_kj = np.ones((K, p))

    for k in range(K):
        denominator = np.zeros(p)
        c = []

        for j in range(p):
            denominator[j] = np.sum(np.diag(V[k]) * ((X_i0[:, j] - g_i[k, j])**2 + (X_s0[:, j] - g_s[k, j])**2))

            if denominator[j] <= epsilon:
                c.append(j)
                lambda_kj[k, j] = lambda_kj_prev[k, j]

        r = p - len(c)
        if r > 0:
            product_h = 1
            for h in range(p):
                if h not in c:
                    product_h *= denominator[h]

            chi = 1 / np.prod([lambda_kj_prev[k, j] for j in c]) if c else 1
            for j in range(p):

                if j not in c:
                    lambda_kj[k, j] = ((chi ** (1 / r)) * (product_h ** (1 / r))) / denominator[j]

    return lambda_kj

def model_fittingV2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):
    test = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))

    lambda_kj = np.ones((K, X_i0.shape[1]))
    lambda_kj_prev = np.ones((K, X_i0.shape[1]))

    J, J_reg, J_cl = compute_JV2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj)

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_c.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertibleV2(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertibleV2(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroidsV2(X_i0, X_s0, V, K)

        lambda_kj = update_lambda_kjV2(X_i0, X_s0, V, g_i, g_s, K, lambda_kj_prev, epsilon)
        lambda_kj_prev = lambda_kj

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):

            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2 +
                alpha*np.sum([lambda_kj[k, j]*((X_i0[i, j] - g_i[k, j])**2 + (X_s0[i, j] - g_s[k, j])**2) for j in range(X_i0.shape[1])]))
                for k in range(K)
            ]

            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1
                test = 1

            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_JV2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj)

        cluster_positions = {k: [] for k in range(K)}

        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)

        beta_c = new_beta_c
        beta_r = new_beta_r

    return beta_c, beta_r, g_i, g_s, lambda_kj, clusters, V

def run_multiple_experiments_V2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):
    best_J = float('inf')
    best_J_reg = None
    best_J_cl = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]

        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, lambda_kj_final, clusters, V_final = model_fittingV2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_JV2(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:], lambda_kj_final)

        if J_final < best_J:
            best_J = J_final
            best_J_reg = J_reg_final
            best_J_cl = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'lambda_j': lambda_kj_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    final_cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(best_result['clusters']):
        final_cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED-LC2} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertibleV7(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_JV7(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj_i, lambda_kj_s):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p_c = X_i0.shape[1]
    p_s = X_s0.shape[1]

    for k in range(K):
        V_k = V[k]

        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum([(X_i0[:, j] - g_i[k, j])**2 * lambda_kj_i[k, j] for j in range(p_c)], axis=0)
        dist_s = np.sum([(X_s0[:, j] - g_s[k, j])**2 * lambda_kj_s[k, j] for j in range(p_s)], axis=0)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroidsV7(X_i0, X_s0, V, K):
    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s


def update_lambda_kjV7(X_i0, X_s0, V, g_i, g_s, K, lambda_kj_i_prev, lambda_kj_s_prev, epsilon):
    p_i = X_i0.shape[1]
    p_s = X_s0.shape[1]

    lambda_kj_i = np.ones((K, X_i0.shape[1]))
    lambda_kj_s = np.ones((K, X_s0.shape[1]))

    for k in range(K):
        c_i = []
        denominator_i = np.zeros(p_i)

        for j in range(p_i):
            denominator_i[j] = np.sum(np.diag(V[k]) * ((X_i0[:, j] - g_i[k, j]) ** 2))
            if denominator_i[j] <= epsilon:
                c_i.append(j)
                lambda_kj_i[k, j] = lambda_kj_i_prev[k, j]

        r_i = p_i - len(c_i)
        if r_i > 0:
            chi_i = 1 / np.prod([lambda_kj_i_prev[k, j] for j in c_i]) if c_i else 1

            denom_nonfrozen = [denominator_i[h] for h in range(p_i) if h not in c_i]
            geom_mean = (np.prod(denom_nonfrozen)) ** (1 / r_i) if denom_nonfrozen else 1

            for j in range(p_i):
                if j not in c_i and denominator_i[j] > epsilon:
                    lambda_kj_i[k, j] = (chi_i ** (1 / r_i)) * geom_mean / denominator_i[j]

        c_s = []
        denominator_s = np.zeros(p_s)

        for j in range(p_s):
            denominator_s[j] = np.sum(np.diag(V[k]) * ((X_s0[:, j] - g_s[k, j]) ** 2))
            if denominator_s[j] <= epsilon:
                c_s.append(j)
                lambda_kj_s[k, j] = lambda_kj_s_prev[k, j]

        r_s = p_s - len(c_s)
        if r_s > 0:
            chi_s = 1 / np.prod([lambda_kj_s_prev[k, j] for j in c_s]) if c_s else 1

            denom_nonfrozen_s = [denominator_s[h] for h in range(p_s) if h not in c_s]
            geom_mean_s = (np.prod(denom_nonfrozen_s)) ** (1 / r_s) if denom_nonfrozen_s else 1

            for j in range(p_s):
                if j not in c_s and denominator_s[j] > epsilon:
                    lambda_kj_s[k, j] = (chi_s ** (1 / r_s)) * geom_mean_s / denominator_s[j]

    return lambda_kj_i, lambda_kj_s


def model_fittingV7(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):
    test = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))

    lambda_kj_i = np.ones((K, X_i0.shape[1]))
    lambda_kj_s = np.ones((K, X_s0.shape[1]))
    lambda_kj_i_prev = np.ones((K, X_i0.shape[1]))
    lambda_kj_s_prev = np.ones((K, X_s0.shape[1]))

    J, J_reg, J_cl = compute_JV7(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj_i, lambda_kj_s)

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_i.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertibleV7(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertibleV7(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroidsV7(X_i0, X_s0, V, K)

        lambda_kj_i, lambda_kj_s = update_lambda_kjV7(X_i0, X_s0, V, g_i, g_s, K, lambda_kj_i_prev, lambda_kj_s_prev, epsilon)
        lambda_kj_i_prev = lambda_kj_i
        lambda_kj_s_prev = lambda_kj_s

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):
            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2 +
                alpha*np.sum([lambda_kj_i[k, j] *(X_i0[i, j] - g_i[k, j])**2 + lambda_kj_s[k,j] * (X_s0[i, j] - g_s[k, j])**2 for j in range(X_i0.shape[1])]))
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1
                test = 1

            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_JV7(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, new_beta_c, new_beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj_i,lambda_kj_s)

        cluster_positions = {k: [] for k in range(K)}

        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)

        beta_c = new_beta_c
        beta_r = new_beta_r

    return beta_c, beta_r, g_i, g_s, lambda_kj_i, lambda_kj_s, clusters, V

def run_multiple_experiments_V7(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):
    best_J = float('inf')
    best_J_reg = None
    best_J_cl = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):
        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, lambda_kj_i_final, lambda_kj_s_final, clusters, V_final = model_fittingV7(
            X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_JV7(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:], lambda_kj_i_final, lambda_kj_s_final)

        if J_final < best_J:
            best_J = J_final
            best_J_reg = J_reg_final
            best_J_cl = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'lambda_kj_i': lambda_kj_i_final,
                'lambda_kj_s': lambda_kj_s_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED-LC3} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertibleV8(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_JV8(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj_i, lambda_kj_s):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p_i = X_i0.shape[1]
    p_s = X_s0.shape[1]

    for k in range(K):
        V_k = V[k]

        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum([(X_i0[:, j] - g_i[k, j])**2 * lambda_kj_i[k, j] for j in range(p_i)], axis=0)
        dist_s = np.sum([(X_s0[:, j] - g_s[k, j])**2 * lambda_kj_s[k, j] for j in range(p_s)], axis=0)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroidsV8(X_i0, X_s0, V, K):
    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s

def update_lambda_kjV8(X_i0, X_s0, V, g_i, g_s, K, lambda_kj_i_prev, lambda_kj_s_prev, epsilon):
    p = X_i0.shape[1]
    lambda_kj_i = np.ones((K, p))
    lambda_kj_s = np.ones((K, p))

    for k in range(K):
        denom_i = np.zeros(p)
        denom_s = np.zeros(p)
        C = []

        for j in range(p):
            denom_i[j] = np.sum(np.diag(V[k]) * (X_i0[:, j] - g_i[k, j])**2)
            denom_s[j] = np.sum(np.diag(V[k]) * (X_s0[:, j] - g_s[k, j])**2)

            if denom_i[j] <= epsilon or denom_s[j] <= epsilon:
                C.append(j)
                lambda_kj_i[k, j] = lambda_kj_i_prev[k, j]
                lambda_kj_s[k, j] = lambda_kj_s_prev[k, j]

        r = p - len(C)
        if r > 0:
            prod_prev = np.prod([lambda_kj_i_prev[k, j] * lambda_kj_s_prev[k, j] for j in C]) if C else 1.0
            if prod_prev > epsilon:
                chi = 1.0 / prod_prev
                exp = 1.0 / (2.0 * r)

                num_shared = 1.0
                for h in range(p):
                    if h not in C:
                        s1 = np.sum(np.diag(V[k]) * (X_i0[:, h] - g_i[k, h])**2)
                        s2 = np.sum(np.diag(V[k]) * (X_s0[:, h] - g_s[k, h])**2)
                        num_shared *= (s1 * s2) ** exp

                for j in range(p):
                    if j not in C:
                        lambda_kj_i[k, j] = (chi ** exp) * num_shared / denom_i[j]
                        lambda_kj_s[k, j] = (chi ** exp) * num_shared / denom_s[j]
            else:
                for j in range(p):
                    lambda_kj_i[k, j] = lambda_kj_i_prev[k, j]
                    lambda_kj_s[k, j] = lambda_kj_s_prev[k, j]

    return lambda_kj_i, lambda_kj_s



def model_fittingV8(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):
    test = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))

    lambda_kj_i = np.ones((K, X_i0.shape[1]))
    lambda_kj_s = np.ones((K, X_s0.shape[1]))
    lambda_kj_i_prev = np.ones((K, X_i0.shape[1]))
    lambda_kj_s_prev = np.ones((K, X_s0.shape[1]))

    J, J_reg, J_cl = compute_JV8(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj_i, lambda_kj_s)

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_c.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertibleV8(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertibleV8(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroidsV8(X_i0, X_s0, V, K)

        lambda_kj_i, lambda_kj_s = update_lambda_kjV8(X_i0, X_s0, V, g_i, g_s, K, lambda_kj_i_prev, lambda_kj_s_prev, epsilon)

        lambda_kj_i_prev = lambda_kj_i
        lambda_kj_s_prev = lambda_kj_s

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):
            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2 +
                alpha*np.sum([lambda_kj_i[k, j] *(X_i0[i, j] - g_i[k, j])**2 + lambda_kj_s[k,j] * (X_s0[i, j] - g_s[k, j])**2 for j in range(X_i0.shape[1])]))
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                test = 1
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1

            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_JV8(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, new_beta_c, new_beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_kj_i,lambda_kj_s)

        cluster_positions = {k: [] for k in range(K)}

        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)

        beta_c = new_beta_c
        beta_r = new_beta_r

    return beta_c, beta_r, g_i, g_s, lambda_kj_i, lambda_kj_s, clusters, V

def run_multiple_experiments_V8(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):
    best_J = float('inf')
    best_J_reg = None
    best_J_cl = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, lambda_kj_i_final, lambda_kj_s_final, clusters, V_final = model_fittingV8(
            X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_JV8(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:], lambda_kj_i_final, lambda_kj_s_final)

        if J_final < best_J:
            best_J = J_final
            best_J_reg = J_reg_final
            best_J_cl = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'lambda_kj_i': lambda_kj_i_final,
                'lambda_kj_s': lambda_kj_s_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED-GL1} \text{ Algorithm}$$**

In [2]:
import numpy as np

def is_invertibleV1(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_JV1(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p = X_i0.shape[1]

    for k in range(K):
        V_k = V[k]

        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum((X_i0 - g_i[k])**2 * lambda_j, axis=1)
        dist_s = np.sum((X_s0 - g_s[k])**2 * lambda_j, axis=1)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroidsV1(X_i0, X_s0, V, K):
    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s

def update_lambda_jV1(X_i0, X_s0, V, g_i, g_s, K, lambda_j_prev, epsilon):
    p = X_i0.shape[1]
    lambda_j = np.ones(p)

    D = np.zeros(p)

    for j in range(p):
        D[j] = sum(
            np.sum(np.diag(V[k]) * (
                (X_i0[:, j] - g_i[k, j])**2 +
                (X_s0[:, j] - g_s[k, j])**2
            ))
            for k in range(K)
        )

    c = [j for j in range(p) if D[j] <= epsilon]

    for j in c:
        lambda_j[j] = lambda_j_prev[j]

    r = p - len(c)

    if r > 0:
        prod_D = np.prod([D[j] for j in range(p) if j not in c])

        geo_mean = prod_D ** (1 / r)

        if len(c) > 0:
            chi = 1 / np.prod([lambda_j_prev[j] for j in c])
        else:
            chi = 1.0

        factor = (chi ** (1 / r)) * geo_mean

        for j in range(p):
            if j not in c:
                lambda_j[j] = factor / D[j]

    return lambda_j

def model_fittingV1(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):
    val = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))

    lambda_j = np.ones(X_i0.shape[1])
    lambda_j_prev = np.ones(X_i0.shape[1])

    J, J_reg, J_cl = compute_JV1(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j)

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_c.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while val != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        val = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertibleV1(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertibleV1(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroidsV1(X_i0, X_s0, V, K)

        lambda_j = update_lambda_jV1(X_i0, X_s0, V, g_i, g_s, K, lambda_j_prev, epsilon)
        lambda_j_prev = lambda_j

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):
            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2) +
                alpha * np.sum(lambda_j * ((X_i0[i] - g_i[k])**2 + (X_s0[i] - g_s[k])**2))
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1
                val = 1

            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_JV1(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, new_beta_c, new_beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j)

        cluster_positions = {k: [] for k in range(K)}

        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)

        beta_c = new_beta_c
        beta_r = new_beta_r

    return beta_c, beta_r, g_i, g_s, lambda_j, clusters, V

def run_multiple_experiments_V1(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):
    best_J = float('inf')
    best_J_reg = None
    best_J_cl = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, lambda_j_final, clusters, V_final = model_fittingV1(
            X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_JV1(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:], lambda_j_final)

        if J_final < best_J:
            best_J = J_final
            best_J_reg = J_reg_final
            best_J_cl = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'lambda_j': lambda_j_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED-GL2} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertibleV5(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_JV5(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j_i, lambda_j_s):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p_i = X_i0.shape[1]
    p_s = X_s0.shape[1]

    for k in range(K):
        V_k = V[k]

        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum((X_i0 - g_i[k])**2 * lambda_j_i, axis=1)
        dist_s = np.sum((X_s0 - g_s[k])**2 * lambda_j_s, axis=1)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroidsV5(X_i0, X_s0, V, K):

    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s

def update_lambda_jV5(X_i0, X_s0, V, g_i, g_s, K, lambda_j_i_prev, lambda_j_s_prev, epsilon):

    p_i = X_i0.shape[1]
    p_s = X_s0.shape[1]

    lambda_j_i = np.ones(p_i)
    lambda_j_s = np.ones(p_s)

    c_i = []
    c_s = []

    for j in range(p_i):
        sum_j_i = np.sum([np.sum(np.diag(V[k]) * ((X_i0[:, j] - g_i[k, j])**2)) for k in range(K)])
        if sum_j_i <= epsilon:
            c_i.append(j)
            lambda_j_i[j] = lambda_j_i_prev[j]

    for j in range(p_s):
        sum_j_s = np.sum([np.sum(np.diag(V[k]) * ((X_s0[:, j] - g_s[k, j])**2)) for k in range(K)])
        if sum_j_s <= epsilon:
            c_s.append(j)
            lambda_j_s[j] = lambda_j_s_prev[j]

    r_i = p_i - len(c_i)
    if r_i > 0:
        chi_i = 1 / np.prod([lambda_j_i_prev[j] for j in c_i]) if c_i else 1
        for j in range(p_i):
            if j not in c_i:
                denom_j = np.sum([np.diag(V[k]) * ((X_i0[:, j] - g_i[k, j])**2) for k in range(K)])
                if denom_j > epsilon:
                    geom_mean = np.prod([
                        (np.sum([np.diag(V[k]) * ((X_i0[:, h] - g_i[k, h])**2) for k in range(K)]))
                        for h in range(p_i) if h not in c_i
                    ]) ** (1 / r_i)
                    lambda_j_i[j] = (chi_i ** (1 / r_i)) * geom_mean / denom_j
                else:
                    lambda_j_i[j] = lambda_j_i_prev[j]

    r_s = p_s - len(c_s)
    if r_s > 0:
        chi_s = 1 / np.prod([lambda_j_s_prev[j] for j in c_s]) if c_s else 1
        for j in range(p_s):
            if j not in c_s:
                denom_j = np.sum([np.diag(V[k]) * ((X_s0[:, j] - g_s[k, j])**2) for k in range(K)])
                if denom_j > epsilon:
                    geom_mean = np.prod([
                        (np.sum([np.diag(V[k]) * ((X_s0[:, h] - g_s[k, h])**2) for k in range(K)]))
                        for h in range(p_s) if h not in c_s
                    ]) ** (1 / r_s)
                    lambda_j_s[j] = (chi_s ** (1 / r_s)) * geom_mean / denom_j
                else:
                    lambda_j_s[j] = lambda_j_s_prev[j]

    return lambda_j_i, lambda_j_s



def model_fittingV5(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):

    test = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))

    lambda_j_i = np.ones(X_i0.shape[1])
    lambda_j_s = np.ones(X_s0.shape[1])
    lambda_j_i_prev = np.ones(X_i0.shape[1])
    lambda_j_s_prev = np.ones(X_s0.shape[1])

    J, J_reg, J_cl = compute_JV5(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j_i, lambda_j_s)

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_i.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertibleV5(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertibleV5(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroidsV5(X_i0, X_s0, V, K)

        lambda_j_i, lambda_j_s = update_lambda_jV5(X_i0, X_s0, V, g_i, g_s, K, lambda_j_i_prev, lambda_j_s_prev, epsilon)
        lambda_j_i_prev = lambda_j_i
        lambda_j_s_prev = lambda_j_s

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):

            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2) +
                alpha * np.sum(lambda_j_i * (X_i0[i] - g_i[k])**2 + lambda_j_s * (X_s0[i] - g_s[k])**2)
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1
                test = 1

            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_JV5(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, new_beta_c, new_beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j_i,lambda_j_s)

        cluster_positions = {k: [] for k in range(K)}

        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)

        beta_c = new_beta_c
        beta_r = new_beta_r

    return beta_c, beta_r, g_i, g_s, lambda_j_i, lambda_j_s, clusters, V

def run_multiple_experiments_V5(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):

    best_J = float('inf')
    best_j_seg = None
    best_j_il = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]

        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, lambda_j_i_final, lambda_j_s_final, clusters, V_final = model_fittingV5(
            X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_JV5(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:], lambda_j_i_final, lambda_j_s_final)

        if J_final < best_J:
            best_J = J_final
            best_j_seg = J_reg_final
            best_j_il = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'lambda_j_i': lambda_j_i_final,
                'lambda_j_s': lambda_j_s_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

### **$$\boldsymbol{J}_{WCLRi-ED-GL3} \text{ Algorithm}$$**

In [ ]:
import numpy as np

def is_invertibleV6(matrix):
    return np.linalg.cond(matrix) < 1 / np.finfo(matrix.dtype).eps

def compute_JV6(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j_i, lambda_j_s):
    J_reg = 0
    J_cl = 0
    N = X_i.shape[0]
    p_i = X_i0.shape[1]
    p_s = X_s0.shape[1]

    for k in range(K):
        V_k = V[k]

        epsilon_c = Y_c - X_c @ beta_c[k]
        epsilon_r = Y_r - X_r @ beta_r[k]
        J_reg += (epsilon_c.T @ V_k @ epsilon_c) + (epsilon_r.T @ V_k @ epsilon_r)

        dist_i = np.sum((X_i0 - g_i[k])**2 * lambda_j_i, axis=1)
        dist_s = np.sum((X_s0 - g_s[k])**2 * lambda_j_s, axis=1)
        J_cl += np.sum(np.diag(V_k) * (dist_i + dist_s))

    J = J_reg + alpha * J_cl
    return J, J_reg, J_cl

def update_centroidsV6(X_i0, X_s0, V, K):
    g_i = np.zeros((K, X_i0.shape[1]))
    g_s = np.zeros((K, X_s0.shape[1]))

    for k in range(K):
        V_k = V[k]
        cluster_indices = np.where(V_k.diagonal() == 1)[0]
        if len(cluster_indices) > 0:
            g_i[k] = np.sum(X_i0[cluster_indices], axis=0) / len(cluster_indices)
            g_s[k] = np.sum(X_s0[cluster_indices], axis=0) / len(cluster_indices)

    return g_i, g_s


def update_lambda_jV6(X_i0, X_s0, V, g_i, g_s, K, lambda_j_i_prev, lambda_j_s_prev, epsilon):
    N, p = X_i0.shape

    lambda_j_i = np.ones(p)
    lambda_j_s = np.ones(p)
    c = []
    denom_i = np.zeros(p)
    denom_s = np.zeros(p)

    for j in range(p):
        denom_i[j] = sum([np.sum(np.diag(V[k]) * ((X_i0[:, j] - g_i[k, j])**2)) for k in range(K)])
        denom_s[j] = sum([np.sum(np.diag(V[k]) * ((X_s0[:, j] - g_s[k, j])**2)) for k in range(K)])

        if denom_i[j] <= epsilon or denom_s[j] <= epsilon:
            c.append(j)
            lambda_j_i[j] = lambda_j_i_prev[j]
            lambda_j_s[j] = lambda_j_s_prev[j]

    r = p - len(c)

    if r > 0:
        prod_frozen = np.prod([lambda_j_i_prev[j] * lambda_j_s_prev[j] for j in c]) if c else 1.0
        chi = 1.0 / prod_frozen
        exp = 1.0 / (2 * r)

        prod_terms = 1.0
        for h in range(p):
            if h not in c:
                s_i = sum([np.sum(np.diag(V[k]) * ((X_i0[:, h] - g_i[k, h])**2)) for k in range(K)])
                s_s = sum([np.sum(np.diag(V[k]) * ((X_s0[:, h] - g_s[k, h])**2)) for k in range(K)])
                prod_terms *= (s_i * s_s) ** exp

        for j in range(p):
            if j not in c:
                s_ij = sum([np.sum(np.diag(V[k]) * (X_i0[:, j] - g_i[k, j])**2) for k in range(K)])
                s_sj = sum([np.sum(np.diag(V[k]) * (X_s0[:, j] - g_s[k, j])**2) for k in range(K)])

                lambda_j_i[j] = (chi**exp) * (prod_terms) / s_ij
                lambda_j_s[j] = (chi**exp) * (prod_terms) / s_sj

    return lambda_j_i, lambda_j_s


def model_fittingV6(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon):

    test = 1
    t = 0

    X_i0 = X_i[:, 1:]
    X_s0 = X_s[:, 1:]

    g_i = np.random.uniform(low=0, high=1.0, size=(K, X_i0.shape[1]))
    g_s = np.random.uniform(low=0, high=1.0, size=(K, X_s0.shape[1]))

    lambda_j_i = np.ones(X_i0.shape[1])
    lambda_j_s = np.ones(X_s0.shape[1])
    lambda_j_i_prev = np.ones(X_i0.shape[1])
    lambda_j_s_prev = np.ones(X_s0.shape[1])

    J, J_reg, J_cl = compute_JV6(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j_i, lambda_j_s)

    initial_clusters = np.array([np.argmax([V[k][i, i] for k in range(K)]) for i in range(X_c.shape[0])])

    cluster_positions = {k: [] for k in range(K)}

    for i, cluster in enumerate(initial_clusters):
        cluster_positions[cluster].append(i + 1)

    while test != 0:
        new_beta_c = np.zeros_like(beta_c)
        new_beta_r = np.zeros_like(beta_r)
        test = 0
        t += 1

        for k in range(K):
            V_k = V[k]

            XtVcXc = X_c.T @ V_k @ X_c
            if is_invertibleV6(XtVcXc):
              new_beta_c[k] = np.linalg.inv(XtVcXc) @ (X_c.T @ V_k @ Y_c)
            else:
              new_beta_c[k] = np.linalg.pinv(XtVcXc) @ (X_c.T @ V_k @ Y_c)

            XtVrXr = X_r.T @ V_k @ X_r
            if is_invertibleV6(XtVrXr):
              new_beta_r[k] = np.linalg.inv(XtVrXr) @ (X_r.T @ V_k @ Y_r)
            else:
              new_beta_r[k] = np.linalg.pinv(XtVrXr) @ (X_r.T @ V_k @ Y_r)

        g_i, g_s = update_centroidsV6(X_i0, X_s0, V, K)

        lambda_j_i, lambda_j_s = update_lambda_jV6(X_i0, X_s0, V, g_i, g_s, K, lambda_j_i_prev, lambda_j_s_prev, epsilon)
        lambda_j_i_prev = lambda_j_i
        lambda_j_s_prev = lambda_j_s

        clusters = np.zeros(X_c.shape[0], dtype=int)

        for i in range(X_c.shape[0]):

            current_cluster = np.argmax([V[k][i, i] for k in range(K)])

            errors = [
                ((Y_c[i] - X_c[i] @ new_beta_c[k])**2 + (Y_r[i] - X_r[i] @ new_beta_r[k])**2) +
                alpha * np.sum(lambda_j_i * (X_i0[i] - g_i[k])**2 + lambda_j_s * (X_s0[i] - g_s[k])**2)
                for k in range(K)
            ]
            new_cluster = np.argmin(errors)

            if new_cluster != current_cluster:
                test = 1
                V[current_cluster][i, i] = 0
                V[new_cluster][i, i] = 1
            clusters[i] = new_cluster

        J, J_reg, J_cl = compute_JV6(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, new_beta_c, new_beta_r, K, alpha, g_i, g_s, X_i0, X_s0, lambda_j_i,lambda_j_s)


        cluster_positions = {k: [] for k in range(K)}

        for i, cluster in enumerate(clusters):
            cluster_positions[cluster].append(i + 1)

        beta_c = new_beta_c
        beta_r = new_beta_r

    return beta_c, beta_r, g_i, g_s, lambda_j_i, lambda_j_s, clusters, V

def run_multiple_experiments_V6(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, K, alpha, r, epsilon):
    best_J = float('inf')
    best_j_seg = None
    best_j_il = None
    best_result = None
    best_iteration = -1

    for rep in range(1, r + 1):

        n_samples = X_c.shape[0]
        V = [np.zeros((n_samples, n_samples)) for _ in range(K)]

        clusters = np.random.randint(0, K, size=n_samples)

        for i in range(n_samples):
            V[clusters[i]][i, i] = 1

        beta_c = np.random.uniform(low=0.0, high=1.0, size=(K, X_c.shape[1]))
        beta_r = np.random.uniform(low=0.0, high=1.0, size=(K, X_r.shape[1]))

        beta_c_final, beta_r_final, g_i_final, g_s_final, lambda_j_i_final, lambda_j_s_final, clusters, V_final = model_fittingV6(
            X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V, beta_c, beta_r, K, alpha, epsilon)

        J_final, J_reg_final, J_cl_final = compute_JV6(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, V_final, beta_c_final, beta_r_final, K, alpha, g_i_final, g_s_final, X_i[:, 1:], X_s[:, 1:], lambda_j_i_final, lambda_j_s_final)

        if J_final < best_J:
            best_J = J_final
            best_j_seg = J_reg_final
            best_j_il = J_cl_final
            best_result = {
                'beta_c': beta_c_final,
                'beta_r': beta_r_final,
                'g_i': g_i_final,
                'g_s': g_s_final,
                'lambda_j_i': lambda_j_i_final,
                'lambda_j_s': lambda_j_s_final,
                'clusters': clusters,
                'V': V_final,
            }
            best_iteration = rep

    cluster_positions = {k: [] for k in range(K)}
    for i, cluster in enumerate(best_result['clusters']):
        cluster_positions[cluster].append(i + 1)

    return best_J, best_result['beta_c'], best_result['beta_r'], best_result['clusters'], best_result['V']

# **Parameter Selection**

## **Cross-Validation**

### **$$\text{Parameter Selection }(\boldsymbol{J}_{CLRi})$$**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np

def kfold_cv_with_knn(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, true_labels, K_values, knn_values, r, epsilon):

    n_samples = X_c.shape[0]
    kf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )
    results = []

    for K in K_values:
        for knn_neighbors in knn_values:

            metrics_val = []
            metrics_train = []

            for train_index, val_index in kf.split(
                X_c,
                true_labels
            ):

                X_c_train, X_r_train = X_c[train_index], X_r[train_index]
                Y_c_train, Y_r_train = Y_c[train_index], Y_r[train_index]
                X_i_train, X_s_train = X_i[train_index], X_s[train_index]
                Y_i_train, Y_s_train = Y_i[train_index], Y_s[train_index]

                X_c_val, X_r_val = X_c[val_index], X_r[val_index]
                Y_c_val, Y_r_val = Y_c[val_index], Y_r[val_index]
                X_i_val, X_s_val = X_i[val_index], X_s[val_index]
                Y_i_val, Y_s_val = Y_i[val_index], Y_s[val_index]

                true_labels_train = true_labels[train_index]

                best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments(
                    X_c_train, X_r_train, Y_c_train, Y_r_train, X_i_train, X_s_train, Y_i_train, Y_s_train, K, r, epsilon
                )

                if K == 1:
                    pred_map = np.zeros(len(X_i_val), dtype=int)
                    pred_map_tr = np.zeros(len(X_i_train), dtype=int)
                    knn_neighbors = None
                else:

                    X_train_combined = np.hstack((X_i_train[:, 1:], X_s_train[:, 1:]))
                    X_val_combined   = np.hstack((X_i_val[:, 1:], X_s_val[:, 1:]))

                    knn = KNeighborsClassifier(
                        n_neighbors=knn_neighbors,
                        metric='euclidean'
                    )

                    knn.fit(X_train_combined, best_cluster)

                    pred_groups = knn.predict(X_val_combined)

                    pred_map = pred_groups

                    pred_map_tr = best_cluster

                Yc_hat = np.array([X_c_val[i] @ best_beta_c[pred_map[i]] for i in range(len(pred_map))])
                Yr_hat = np.array([X_r_val[i] @ best_beta_r[pred_map[i]] for i in range(len(pred_map))])

                Yi_hat = Yc_hat - Yr_hat
                Ys_hat = Yc_hat + Yr_hat

                rmseL = np.sqrt(mean_squared_error(Y_i_val, Yi_hat))
                rmseU = np.sqrt(mean_squared_error(Y_s_val, Ys_hat))
                rmseLU = np.sqrt(np.mean((Y_i_val-Yi_hat)**2 + (Y_s_val-Ys_hat)**2))

                rmseC = np.sqrt(mean_squared_error(Y_c_val, Yc_hat))
                rmseR = np.sqrt(mean_squared_error(Y_r_val, Yr_hat))
                rmseCR = np.sqrt(np.mean((Y_c_val-Yc_hat)**2 + (Y_r_val-Yr_hat)**2))

                metrics_val.append([rmseL,rmseU,rmseLU,rmseC,rmseR,rmseCR])

                Yc_hat_tr = np.array([X_c_train[i] @ best_beta_c[pred_map_tr[i]] for i in range(len(pred_map_tr))])
                Yr_hat_tr = np.array([X_r_train[i] @ best_beta_r[pred_map_tr[i]] for i in range(len(pred_map_tr))])

                Yi_hat_tr = Yc_hat_tr - Yr_hat_tr
                Ys_hat_tr = Yc_hat_tr + Yr_hat_tr

                rmseLt = np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr))
                rmseUt = np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr))
                rmseLUt = np.sqrt(np.mean((Y_i_train-Yi_hat_tr)**2 + (Y_s_train-Ys_hat_tr)**2))

                rmseCt = np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr))
                rmseRt = np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr))
                rmseCRt = np.sqrt(np.mean((Y_c_train-Yc_hat_tr)**2 + (Y_r_train-Yr_hat_tr)**2))

                metrics_train.append([rmseLt,rmseUt,rmseLUt,rmseCt,rmseRt,rmseCRt])

            metrics_val = np.mean(metrics_val, axis=0)
            metrics_train = np.mean(metrics_train, axis=0)

            results.append({
                'K':K,'knn':knn_neighbors,
                'val':metrics_val,
                'train':metrics_train
            })

    best = sorted(results, key=lambda x: x['val'][-1])[0]

    print("\n" + "="*50)
    print("Best Parameters")
    print("="*50)

    print(f"K (clusters): {best['K']}")
    print(f"KNN (NEIGHBORS): {best['knn']}")

    # =========================
    #  VALIDATION
    # =========================
    val = best['val']

    print("\n METRICS - VALIDATION")
    print("-"*50)


    print(f"RMSE Lower (RMSE_L): {val[0]:.4f}")
    print(f"RMSE Upper (RMSE_U): {val[1]:.4f}")
    print(f"RMSE Interval (RMSE_LU): {val[2]:.4f}")

    print(f"RMSE CENTRE (RMSE_C): {val[3]:.4f}")
    print(f"RMSE RANGE (RMSE_R): {val[4]:.4f}")
    print(f"RMSE COMBINED (RMSE_CR): {val[5]:.4f}")

    # =========================
    #  TREINO
    # =========================
    train = best['train']

    print("\n METRICS - TRAIN")
    print("-"*50)

    print(f"RMSE Lower (RMSE_L): {train[0]:.4f}")
    print(f"RMSE Upper (RMSE_U): {train[1]:.4f}")
    print(f"RMSE Interval (RMSE_LU): {train[2]:.4f}")

    print(f"RMSE CENTRE (RMSE_C): {train[3]:.4f}")
    print(f"RMSE RANGE (RMSE_R): {train[4]:.4f}")
    print(f"RMSE COMBINED (RMSE_CR): {train[5]:.4f}")
    print()
    print()
    return best['K'], best['knn']


Tempo de execução: 00:00:00.74 (hh:mm:ss)


### **$$\text{Parameter Selection }(\boldsymbol{J}_{WCLRi} \text{ and Variants})$$**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np

def kfold_cv_with_knn_V(X_c, X_r, Y_c, Y_r, X_i, X_s, Y_i, Y_s, true_labels, K_values, knn_values, alpha_values, r, epsilon):

    n_samples = X_c.shape[0]
    kf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )
    results = []

    for K in K_values:
        for alpha in alpha_values:
            for knn_neighbors in knn_values:

                metrics_val = []
                metrics_train = []

                for train_index, val_index in kf.split(
                    X_c,
                    true_labels
                ):

                    X_c_train, X_r_train = X_c[train_index], X_r[train_index]
                    Y_c_train, Y_r_train = Y_c[train_index], Y_r[train_index]
                    X_i_train, X_s_train = X_i[train_index], X_s[train_index]
                    Y_i_train, Y_s_train = Y_i[train_index], Y_s[train_index]

                    X_c_val, X_r_val = X_c[val_index], X_r[val_index]
                    Y_c_val, Y_r_val = Y_c[val_index], Y_r[val_index]
                    X_i_val, X_s_val = X_i[val_index], X_s[val_index]
                    Y_i_val, Y_s_val = Y_i[val_index], Y_s[val_index]

                    true_labels_train = true_labels[train_index]

                    best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments_V(
                        X_c_train, X_r_train, Y_c_train, Y_r_train,
                        X_i_train, X_s_train, Y_i_train, Y_s_train,
                        K, alpha, r, epsilon
                    )

                    if K == 1:
                        pred_map = np.zeros(len(X_i_val), dtype=int)
                        pred_map_tr = np.zeros(len(X_i_train), dtype=int)
                        knn_neighbors = None
                    else:
                        X_train_combined = np.hstack((X_i_train[:, 1:], X_s_train[:, 1:]))
                        X_val_combined   = np.hstack((X_i_val[:, 1:], X_s_val[:, 1:]))

                        knn = KNeighborsClassifier(
                            n_neighbors=knn_neighbors,
                            metric='euclidean'
                        )

                        knn.fit(X_train_combined, best_cluster)

                        pred_groups = knn.predict(X_val_combined)

                        pred_map = pred_groups

                        pred_map_tr = best_cluster

                    Yc_hat = np.array([X_c_val[i] @ best_beta_c[pred_map[i]] for i in range(len(pred_map))])
                    Yr_hat = np.array([X_r_val[i] @ best_beta_r[pred_map[i]] for i in range(len(pred_map))])

                    Yi_hat = Yc_hat - Yr_hat
                    Ys_hat = Yc_hat + Yr_hat

                    rmseL = np.sqrt(mean_squared_error(Y_i_val, Yi_hat))
                    rmseU = np.sqrt(mean_squared_error(Y_s_val, Ys_hat))
                    rmseLU = np.sqrt(np.mean((Y_i_val-Yi_hat)**2 + (Y_s_val-Ys_hat)**2))

                    rmseC = np.sqrt(mean_squared_error(Y_c_val, Yc_hat))
                    rmseR = np.sqrt(mean_squared_error(Y_r_val, Yr_hat))
                    rmseCR = np.sqrt(np.mean((Y_c_val-Yc_hat)**2 + (Y_r_val-Yr_hat)**2))

                    metrics_val.append([rmseL,rmseU,rmseLU,rmseC,rmseR,rmseCR])

                    Yc_hat_tr = np.array([X_c_train[i] @ best_beta_c[pred_map_tr[i]] for i in range(len(pred_map_tr))])
                    Yr_hat_tr = np.array([X_r_train[i] @ best_beta_r[pred_map_tr[i]] for i in range(len(pred_map_tr))])

                    Yi_hat_tr = Yc_hat_tr - Yr_hat_tr
                    Ys_hat_tr = Yc_hat_tr + Yr_hat_tr

                    rmseLt = np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr))
                    rmseUt = np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr))
                    rmseLUt = np.sqrt(np.mean((Y_i_train-Yi_hat_tr)**2 + (Y_s_train-Ys_hat_tr)**2))

                    rmseCt = np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr))
                    rmseRt = np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr))
                    rmseCRt = np.sqrt(np.mean((Y_c_train-Yc_hat_tr)**2 + (Y_r_train-Yr_hat_tr)**2))

                    metrics_train.append([rmseLt,rmseUt,rmseLUt,rmseCt,rmseRt,rmseCRt])

                metrics_val = np.mean(metrics_val, axis=0)
                metrics_train = np.mean(metrics_train, axis=0)

                results.append({
                    'K':K,'knn':knn_neighbors,'alpha':alpha,
                    'val':metrics_val,
                    'train':metrics_train
                })

    best = sorted(results, key=lambda x: x['val'][-1])[0]

    print("\n" + "="*50)
    print("Best Parameters")
    print("="*50)

    print(f"K (clusters): {best['K']}")
    print(f"KNN (NEIGHBORS): {best['knn']}")
    print(f"Alpha: {best['alpha']}")

    # =========================
    #  VALIDAÇÃO
    # =========================
    val = best['val']

    print("\n METRICS - VALIDATION")
    print("-"*50)


    print(f"RMSE Lower (RMSE_L): {val[0]:.4f}")
    print(f"RMSE Upper (RMSE_U): {val[1]:.4f}")
    print(f"RMSE Interval (RMSE_LU): {val[2]:.4f}")

    print(f"RMSE CENTRE (RMSE_C): {val[3]:.4f}")
    print(f"RMSE RANGE (RMSE_R): {val[4]:.4f}")
    print(f"RMSE COMBINED (RMSE_CR): {val[5]:.4f}")

    # =========================
    #  TREINO
    # =========================
    train = best['train']

    print("\n METRICS - TRAIN")
    print("-"*50)

    print(f"RMSE Lower (RMSE_L): {train[0]:.4f}")
    print(f"RMSE Upper (RMSE_U): {train[1]:.4f}")
    print(f"RMSE Interval (RMSE_LU): {train[2]:.4f}")

    print(f"RMSE CENTRE (RMSE_C): {train[3]:.4f}")
    print(f"RMSE RANGE (RMSE_R): {train[4]:.4f}")
    print(f"RMSE COMBINED (RMSE_CR): {train[5]:.4f}")
    print()
    print()
    return best['K'], best['knn'], best['alpha']

## **Bootstrap .632+**

### **$$\text{Parameter Selection }(\boldsymbol{J}_{CLRi})$$**

In [ ]:
from sklearn.utils import resample
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error
import numpy as np

# =========================================================
#  BOOTSTRAP .632+ FUNCTION
# =========================================================

def bootstrap_632_plus(err_train, err_oob, err_noinfo):

    denom = (err_noinfo - err_train) + 1e-12

    R = (err_oob - err_train) / denom

    R = np.clip(R, 0, 1)

    w = 0.632 / (1 - 0.368 * R)

    err_632_plus = (1 - w)*err_train + w*err_oob

    return err_632_plus

# =========================================================
#  MAIN FUNCTION
# =========================================================

def bootstrap_cv_with_knn(
    X_c, X_r, Y_c, Y_r,
    X_i, X_s, Y_i, Y_s,
    K_values, knn_values,
    r, epsilon,
    B=100
):

    n_samples = X_c.shape[0]

    results = []

    for K in K_values:

        for knn_neighbors in knn_values:

            metrics_train = []
            metrics_oob = []
            metrics_632 = []
            metrics_632plus = []

            # =========================================================
            # BOOTSTRAP
            # =========================================================

            for b in range(B):

                train_index = resample(
                    np.arange(n_samples),
                    replace=True,
                    n_samples=n_samples
                )

                mask = np.ones(n_samples, dtype=bool)
                mask[train_index] = False

                val_index = np.where(mask)[0]

                if len(val_index) == 0:
                    continue

                # =========================================================
                #  SPLIT
                # =========================================================

                X_c_train = X_c[train_index]
                X_r_train = X_r[train_index]

                X_i_train = X_i[train_index]
                X_s_train = X_s[train_index]

                Y_c_train = Y_c[train_index]
                Y_r_train = Y_r[train_index]

                Y_i_train = Y_i[train_index]
                Y_s_train = Y_s[train_index]

                X_c_val = X_c[val_index]
                X_r_val = X_r[val_index]

                X_i_val = X_i[val_index]
                X_s_val = X_s[val_index]

                Y_c_val = Y_c[val_index]
                Y_r_val = Y_r[val_index]

                Y_i_val = Y_i[val_index]
                Y_s_val = Y_s[val_index]

                # =========================================================
                #  TRAIN
                # =========================================================

                best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments(
                    X_c_train, X_r_train,
                    Y_c_train, Y_r_train,
                    X_i_train, X_s_train,
                    Y_i_train, Y_s_train,
                    K, r, epsilon
                )

                # =========================================================
                #  KNN
                # =========================================================

                if K == 1:

                    pred_map = np.zeros(len(X_i_val), dtype=int)
                    pred_map_tr = np.zeros(len(X_i_train), dtype=int)

                else:

                    X_train_combined = np.hstack((
                        X_i_train[:,1:],
                        X_s_train[:,1:]
                    ))

                    X_val_combined = np.hstack((
                        X_i_val[:,1:],
                        X_s_val[:,1:]
                    ))

                    knn = KNeighborsClassifier(
                        n_neighbors=knn_neighbors,
                        metric='euclidean'
                    )

                    knn.fit(X_train_combined, best_cluster)

                    pred_map = knn.predict(X_val_combined)
                    pred_map_tr = knn.predict(X_train_combined)

                # =========================================================
                #  PREDICTION - OOB
                # =========================================================

                Yc_hat = np.array([
                    X_c_val[i] @ best_beta_c[pred_map[i]]
                    for i in range(len(pred_map))
                ])

                Yr_hat = np.array([
                    X_r_val[i] @ best_beta_r[pred_map[i]]
                    for i in range(len(pred_map))
                ])

                Yi_hat = Yc_hat - Yr_hat
                Ys_hat = Yc_hat + Yr_hat

                # =========================================================
                #  PREDICTION - TRAIN
                # =========================================================

                Yc_hat_tr = np.array([
                    X_c_train[i] @ best_beta_c[pred_map_tr[i]]
                    for i in range(len(pred_map_tr))
                ])

                Yr_hat_tr = np.array([
                    X_r_train[i] @ best_beta_r[pred_map_tr[i]]
                    for i in range(len(pred_map_tr))
                ])

                Yi_hat_tr = Yc_hat_tr - Yr_hat_tr
                Ys_hat_tr = Yc_hat_tr + Yr_hat_tr

                # =========================================================
                #  RMSE TRAIN
                # =========================================================

                rmseLt = np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr))
                rmseUt = np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr))
                rmseLUt = np.sqrt(np.mean(
                    (Y_i_train - Yi_hat_tr)**2 +
                    (Y_s_train - Ys_hat_tr)**2
                ))

                rmseCt = np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr))
                rmseRt = np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr))
                rmseCRt = np.sqrt(np.mean(
                    (Y_c_train - Yc_hat_tr)**2 +
                    (Y_r_train - Yr_hat_tr)**2
                ))

                # =========================================================
                #  RMSE OOB
                # =========================================================

                rmseL = np.sqrt(mean_squared_error(Y_i_val, Yi_hat))
                rmseU = np.sqrt(mean_squared_error(Y_s_val, Ys_hat))
                rmseLU = np.sqrt(np.mean(
                    (Y_i_val - Yi_hat)**2 +
                    (Y_s_val - Ys_hat)**2
                ))

                rmseC = np.sqrt(mean_squared_error(Y_c_val, Yc_hat))
                rmseR = np.sqrt(mean_squared_error(Y_r_val, Yr_hat))
                rmseCR = np.sqrt(np.mean(
                    (Y_c_val - Yc_hat)**2 +
                    (Y_r_val - Yr_hat)**2
                ))

                train_metrics = np.array([
                    rmseLt,
                    rmseUt,
                    rmseLUt,
                    rmseCt,
                    rmseRt,
                    rmseCRt
                ])

                oob_metrics = np.array([
                    rmseL,
                    rmseU,
                    rmseLU,
                    rmseC,
                    rmseR,
                    rmseCR
                ])

                # =========================================================
                #  NO-INFORMATION ERROR
                # =========================================================

                baseline_L = np.mean(Y_i_train)
                baseline_U = np.mean(Y_s_train)

                baseline_C = np.mean(Y_c_train)
                baseline_R = np.mean(Y_r_train)

                rmseL_noinfo = np.sqrt(mean_squared_error(
                    Y_i_val,
                    np.full(len(Y_i_val), baseline_L)
                ))

                rmseU_noinfo = np.sqrt(mean_squared_error(
                    Y_s_val,
                    np.full(len(Y_s_val), baseline_U)
                ))

                rmseC_noinfo = np.sqrt(mean_squared_error(
                    Y_c_val,
                    np.full(len(Y_c_val), baseline_C)
                ))

                rmseR_noinfo = np.sqrt(mean_squared_error(
                    Y_r_val,
                    np.full(len(Y_r_val), baseline_R)
                ))

                rmseLU_noinfo = np.sqrt(
                    rmseL_noinfo**2 +
                    rmseU_noinfo**2
                )

                rmseCR_noinfo = np.sqrt(
                    rmseC_noinfo**2 +
                    rmseR_noinfo**2
                )

                noinfo_metrics = np.array([
                    rmseL_noinfo,
                    rmseU_noinfo,
                    rmseLU_noinfo,
                    rmseC_noinfo,
                    rmseR_noinfo,
                    rmseCR_noinfo
                ])

                # =========================================================
                #  BOOTSTRAP .632
                # =========================================================

                metrics632 = (
                    0.368 * train_metrics +
                    0.632 * oob_metrics
                )

                # =========================================================
                #  BOOTSTRAP .632+
                # =========================================================

                metrics632plus = np.array([

                    bootstrap_632_plus(
                        train_metrics[i],
                        oob_metrics[i],
                        noinfo_metrics[i]
                    )

                    for i in range(len(train_metrics))

                ])

                metrics_train.append(train_metrics)
                metrics_oob.append(oob_metrics)

                metrics_632.append(metrics632)
                metrics_632plus.append(metrics632plus)

            # =========================================================
            #  MEAN
            # =========================================================

            metrics_train = np.mean(metrics_train, axis=0)
            metrics_oob = np.mean(metrics_oob, axis=0)

            metrics_632 = np.mean(metrics_632, axis=0)
            metrics_632plus = np.mean(metrics_632plus, axis=0)

            results.append({

                'K': K,
                'knn': knn_neighbors,

                'train': metrics_train,
                'oob': metrics_oob,

                'b632': metrics_632,
                'b632plus': metrics_632plus

            })

    # =========================================================
    # BEST MODEL THROUGH RMSE_CR .632+
    # =========================================================

    best = sorted(
        results,
        key=lambda x: x['b632plus'][-1]
    )[0]

    # =========================================================
    #  RESULTS
    # =========================================================

    print("\n" + "="*60)
    print("Best Parameters")
    print("="*60)

    print(f"K: {best['K']}")
    print(f"KNN: {best['knn']}")

    nomes = [
        "RMSE_L",
        "RMSE_U",
        "RMSE_LU",
        "RMSE_C",
        "RMSE_R",
        "RMSE_CR"
    ]

    grupos = {
        "TREINO": best['train'],
        "BOOTSTRAP/OOB": best['oob'],
        "BOOTSTRAP .632": best['b632'],
        "BOOTSTRAP .632+": best['b632plus']
    }


    return best['K'], best['knn']

### **$$\text{Parameter Selection }(\boldsymbol{J}_{WCLRi} \text{ and Variants})$$**

In [ ]:
from sklearn.utils import resample
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error
import numpy as np

# =========================================================
#  BOOTSTRAP .632+ FUNCTION
# =========================================================

def bootstrap_632_plus(err_train, err_oob, err_noinfo):

    denom = (err_noinfo - err_train) + 1e-12

    R = (err_oob - err_train) / denom

    R = np.clip(R, 0, 1)

    w = 0.632 / (1 - 0.368 * R)

    err_632_plus = (1 - w)*err_train + w*err_oob

    return err_632_plus


# =========================================================
#  MAIN FUNCTION
# =========================================================

def bootstrap_cv_with_knn_v(
    X_c, X_r, Y_c, Y_r,
    X_i, X_s, Y_i, Y_s,
    K_values, knn_values, alpha_values,
    r, epsilon,
    B=100
):

    n_samples = X_c.shape[0]

    results = []

    for K in K_values:

        for alpha in alpha_values:

            for knn_neighbors in knn_values:

                metrics_train = []
                metrics_oob = []
                metrics_632 = []
                metrics_632plus = []

                # =========================================================
                # BOOTSTRAP
                # =========================================================

                for b in range(B):

                    train_index = resample(
                        np.arange(n_samples),
                        replace=True,
                        n_samples=n_samples
                    )

                    mask = np.ones(n_samples, dtype=bool)
                    mask[train_index] = False

                    val_index = np.where(mask)[0]

                    if len(val_index) == 0:
                        continue

                    # =========================================================
                    #  SPLIT
                    # =========================================================

                    X_c_train = X_c[train_index]
                    X_r_train = X_r[train_index]

                    X_i_train = X_i[train_index]
                    X_s_train = X_s[train_index]

                    Y_c_train = Y_c[train_index]
                    Y_r_train = Y_r[train_index]

                    Y_i_train = Y_i[train_index]
                    Y_s_train = Y_s[train_index]

                    X_c_val = X_c[val_index]
                    X_r_val = X_r[val_index]

                    X_i_val = X_i[val_index]
                    X_s_val = X_s[val_index]

                    Y_c_val = Y_c[val_index]
                    Y_r_val = Y_r[val_index]

                    Y_i_val = Y_i[val_index]
                    Y_s_val = Y_s[val_index]

                    # =========================================================
                    #  TRAIN
                    # =========================================================

                    best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments_V(
                        X_c_train, X_r_train,
                        Y_c_train, Y_r_train,
                        X_i_train, X_s_train,
                        Y_i_train, Y_s_train,
                        K, alpha,
                        r, epsilon
                    )

                    # =========================================================
                    #  KNN
                    # =========================================================

                    if K == 1:

                        pred_map = np.zeros(len(X_i_val), dtype=int)
                        pred_map_tr = np.zeros(len(X_i_train), dtype=int)

                    else:

                        from sklearn.preprocessing import StandardScaler

                        scaler = StandardScaler()

                        X_train_combined0 = np.hstack((
                            X_i_train[:,1:],
                            X_s_train[:,1:]
                        ))

                        X_val_combined0 = np.hstack((
                            X_i_val[:,1:],
                            X_s_val[:,1:]
                        ))

                        X_train_combined = scaler.fit_transform(X_train_combined0)

                        X_val_combined = scaler.transform(X_val_combined0)
                        knn = KNeighborsClassifier(
                            n_neighbors=knn_neighbors,
                            metric='euclidean'
                        )

                        knn.fit(X_train_combined, best_cluster)

                        pred_map = knn.predict(X_val_combined)
                        pred_map_tr = knn.predict(X_train_combined)

                    # =========================================================
                    #  PREDICTION OOB
                    # =========================================================

                    Yc_hat = np.array([
                        X_c_val[i] @ best_beta_c[pred_map[i]]
                        for i in range(len(pred_map))
                    ])

                    Yr_hat = np.array([
                        X_r_val[i] @ best_beta_r[pred_map[i]]
                        for i in range(len(pred_map))
                    ])

                    Yi_hat = Yc_hat - Yr_hat
                    Ys_hat = Yc_hat + Yr_hat

                    # =========================================================
                    #  PREDICTION TRAIN
                    # =========================================================

                    Yc_hat_tr = np.array([
                        X_c_train[i] @ best_beta_c[pred_map_tr[i]]
                        for i in range(len(pred_map_tr))
                    ])

                    Yr_hat_tr = np.array([
                        X_r_train[i] @ best_beta_r[pred_map_tr[i]]
                        for i in range(len(pred_map_tr))
                    ])

                    Yi_hat_tr = Yc_hat_tr - Yr_hat_tr
                    Ys_hat_tr = Yc_hat_tr + Yr_hat_tr

                    # =========================================================
                    #  RMSE TRAIN
                    # =========================================================

                    rmseLt = np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr))
                    rmseUt = np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr))
                    rmseLUt = np.sqrt(np.mean(
                        (Y_i_train - Yi_hat_tr)**2 +
                        (Y_s_train - Ys_hat_tr)**2
                    ))

                    rmseCt = np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr))
                    rmseRt = np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr))
                    rmseCRt = np.sqrt(np.mean(
                        (Y_c_train - Yc_hat_tr)**2 +
                        (Y_r_train - Yr_hat_tr)**2
                    ))

                    # =========================================================
                    #  RMSE OOB
                    # =========================================================

                    rmseL = np.sqrt(mean_squared_error(Y_i_val, Yi_hat))
                    rmseU = np.sqrt(mean_squared_error(Y_s_val, Ys_hat))
                    rmseLU = np.sqrt(np.mean(
                        (Y_i_val - Yi_hat)**2 +
                        (Y_s_val - Ys_hat)**2
                    ))

                    rmseC = np.sqrt(mean_squared_error(Y_c_val, Yc_hat))
                    rmseR = np.sqrt(mean_squared_error(Y_r_val, Yr_hat))
                    rmseCR = np.sqrt(np.mean(
                        (Y_c_val - Yc_hat)**2 +
                        (Y_r_val - Yr_hat)**2
                    ))

                    train_metrics = np.array([
                        rmseLt,
                        rmseUt,
                        rmseLUt,
                        rmseCt,
                        rmseRt,
                        rmseCRt
                    ])

                    oob_metrics = np.array([
                        rmseL,
                        rmseU,
                        rmseLU,
                        rmseC,
                        rmseR,
                        rmseCR
                    ])

                    # =========================================================
                    #  NO-INFORMATION ERROR
                    # =========================================================

                    baseline_L = np.mean(Y_i_train)
                    baseline_U = np.mean(Y_s_train)

                    baseline_C = np.mean(Y_c_train)
                    baseline_R = np.mean(Y_r_train)

                    rmseL_noinfo = np.sqrt(mean_squared_error(
                        Y_i_val,
                        np.full(len(Y_i_val), baseline_L)
                    ))

                    rmseU_noinfo = np.sqrt(mean_squared_error(
                        Y_s_val,
                        np.full(len(Y_s_val), baseline_U)
                    ))

                    rmseC_noinfo = np.sqrt(mean_squared_error(
                        Y_c_val,
                        np.full(len(Y_c_val), baseline_C)
                    ))

                    rmseR_noinfo = np.sqrt(mean_squared_error(
                        Y_r_val,
                        np.full(len(Y_r_val), baseline_R)
                    ))

                    rmseLU_noinfo = np.sqrt(
                        rmseL_noinfo**2 +
                        rmseU_noinfo**2
                    )

                    rmseCR_noinfo = np.sqrt(
                        rmseC_noinfo**2 +
                        rmseR_noinfo**2
                    )

                    noinfo_metrics = np.array([
                        rmseL_noinfo,
                        rmseU_noinfo,
                        rmseLU_noinfo,
                        rmseC_noinfo,
                        rmseR_noinfo,
                        rmseCR_noinfo
                    ])

                    # =========================================================
                    #  BOOTSTRAP .632
                    # =========================================================

                    metrics632 = (
                        0.368 * train_metrics +
                        0.632 * oob_metrics
                    )

                    # =========================================================
                    #  BOOTSTRAP .632+
                    # =========================================================

                    metrics632plus = np.array([

                        bootstrap_632_plus(
                            train_metrics[i],
                            oob_metrics[i],
                            noinfo_metrics[i]
                        )

                        for i in range(len(train_metrics))

                    ])

                    metrics_train.append(train_metrics)
                    metrics_oob.append(oob_metrics)

                    metrics_632.append(metrics632)
                    metrics_632plus.append(metrics632plus)

                # =========================================================
                #  MEAN
                # =========================================================

                metrics_train = np.mean(metrics_train, axis=0)
                metrics_oob = np.mean(metrics_oob, axis=0)

                metrics_632 = np.mean(metrics_632, axis=0)
                metrics_632plus = np.mean(metrics_632plus, axis=0)

                results.append({

                    'K': K,
                    'knn': knn_neighbors,
                    'alpha': alpha,

                    'train': metrics_train,
                    'oob': metrics_oob,

                    'b632': metrics_632,
                    'b632plus': metrics_632plus

                })

    # =========================================================
    # BEST MODEL THROUGH RMSE_CR .632+
    # =========================================================

    best = sorted(
        results,
        key=lambda x: x['b632plus'][-1]
    )[0]

    # =========================================================
    #  RESULTS
    # =========================================================

    print("\n" + "="*60)
    print("Best Parameters")
    print("="*60)

    print(f"K: {best['K']}")
    print(f"KNN: {best['knn']}")
    print(f"Alpha: {best['alpha']}")

    nomes = [
        "RMSE_L",
        "RMSE_U",
        "RMSE_LU",
        "RMSE_C",
        "RMSE_R",
        "RMSE_CR"
    ]

    grupos = {
        "TREINO": best['train'],
        "BOOTSTRAP/OOB": best['oob'],
        "BOOTSTRAP .632": best['b632'],
        "BOOTSTRAP .632+": best['b632plus']
    }

    return best['K'], best['knn'], best['alpha']

# **Overall Algortithm**

## **Cross-Validation**

### **$$\text{Overall }\boldsymbol{J}_{CLRi}$$**

In [ ]:
%%time
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np

def interval_metrics(Y_i, Y_s, Y_i_hat, Y_s_hat):
    w_int = np.maximum(0, np.minimum(Y_s, Y_s_hat) - np.maximum(Y_i, Y_i_hat))
    w_union = np.maximum(Y_s, Y_s_hat) - np.minimum(Y_i, Y_i_hat)
    w_y = Y_s - Y_i
    w_yhat = Y_s_hat - Y_i_hat

    CR = np.mean(w_int / (w_y + 1e-12))
    ER = np.mean(w_int / (w_yhat + 1e-12))
    RI = np.mean(w_int / (w_union + 1e-12))

    return CR, ER, RI

def r2_interval(y, yhat):
    if np.std(y) == 0 or np.std(yhat) == 0:
        return 0
    cov = np.cov(y, yhat, bias=True)[0,1]
    return (cov / (np.std(y) * np.std(yhat)))**2

# =========================
#  PARAMETERS
# =========================

K_values = range(3,4)
epsilon = 1e-6
r = 100  # RANDOM INICIALIZATION
knn_values = range(1,18)
n_mc = 50

# =========================
#  RESULT LIST
# =========================

# RMSE
all_rmse_L_test, all_rmse_U_test, all_rmse_LU_test = [], [], []
all_rmse_L_train, all_rmse_U_train, all_rmse_LU_train = [], [], []

all_rmse_c_test, all_rmse_r_test, all_rmse_conj_c_r_test = [], [], []
all_rmse_c_train, all_rmse_r_train, all_rmse_conj_c_r_train = [], [], []

# Interval
all_CR_test, all_ER_test, all_RI_test = [], [], []
all_CR_train, all_ER_train, all_RI_train = [], [], []

# Correlation
all_r2L_test, all_r2U_test, all_r2LU_test = [], [], []
all_r2C_test, all_r2R_test, all_r2CR_test = [], [], []

all_r2L_train, all_r2U_train, all_r2LU_train = [], [], []
all_r2C_train, all_r2R_train, all_r2CR_train = [], [], []

# Cluster
all_ari_test, all_nmi_test, all_ari_tr, all_nmi_tr = [], [], [], []

# =========================
#  MONTE CARLO
# =========================
for mc in range(n_mc):
    print(f"\n Monte Carlo {mc+1}/{n_mc}")

    df = generate_dataset1(n_per_class=100, seed=(mc+10))
    true_labels = df['Classe'].values

    X_i0 = df[['X1_min', 'X2_min']].values
    X_s0 = df[['X1_max', 'X2_max']].values
    Y_i = df['Y_min'].values
    Y_s = df['Y_max'].values

    X_c0 = df[['X1_c', 'X2_c']].values
    X_r0 = df[['X1_r', 'X2_r']].values
    Y_c = df['Y_c'].values
    Y_r = df['Y_r'].values

    n = len(df)
    intercept = np.ones((n,1))

    X_i = np.hstack((intercept, X_i0))
    X_s = np.hstack((intercept, X_s0))
    X_c = np.hstack((intercept, X_c0))
    X_r = np.hstack((intercept, X_r0))

    # =========================
    #  SPLIT
    # =========================
    X_c_train, X_c_test, X_r_train, X_r_test, \
    X_i_train, X_i_test, X_s_train, X_s_test, \
    Y_c_train, Y_c_test, Y_r_train, Y_r_test, \
    Y_i_train, Y_i_test, Y_s_train, Y_s_test, \
    true_labels_train, true_labels_test = train_test_split(
        X_c, X_r, X_i, X_s,
        Y_c, Y_r, Y_i, Y_s,
        true_labels,

        test_size=0.2,

        stratify=true_labels,

        random_state=mc
    )

    # =========================
    #  K-FOLD
    # =========================
    best_K, best_knn = kfold_cv_with_knn(
        X_c_train, X_r_train, Y_c_train, Y_r_train,
        X_i_train, X_s_train, Y_i_train, Y_s_train,
        true_labels_train, K_values, knn_values, r, epsilon
    )

    # =========================
    #  FINAL TRAIN
    # =========================
    best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments(
        X_c_train, X_r_train, Y_c_train, Y_r_train,
        X_i_train, X_s_train, Y_i_train, Y_s_train,
        best_K, r, epsilon
    )

    print("\n" + "="*80)
    print(f"RESULTS - EXPERIMENT {mc+1}")
    print("="*80)

    for k in range(len(best_beta_c)):
        n_k = np.sum(best_cluster == k)
        print(f"Group {k} (n={n_k})")
        print(f"  beta_c: {np.round(best_beta_c[k], 4)}")
        print(f"  beta_r: {np.round(best_beta_r[k], 4)}")
        print()

    # =========================
    #  PEDICTION
    # =========================
    if best_K == 1:
        best_knn = None
        pred_map = np.zeros(len(X_i_test), dtype=int)
        pred_map_tr = np.zeros(len(X_i_train), dtype=int)

    else:
        X_train_combined = np.hstack((X_c_train[:, 1:], X_r_train[:, 1:]))
        X_test_combined   = np.hstack((X_c_test[:, 1:],   X_r_test[:, 1:]))

        knn = KNeighborsClassifier(
            n_neighbors=best_knn,
            metric='euclidean'
        )

        knn.fit(X_train_combined, best_cluster)

        # =========================
        #  TEST
        # =========================
        pred_groups = knn.predict(X_test_combined)

        pred_map = pred_groups

        # =========================
        #  TRAIN
        # =========================
        pred_map_tr = best_cluster

    Yc_hat = np.array([X_c_test[i] @ best_beta_c[pred_map[i]] for i in range(len(pred_map))])
    Yr_hat = np.array([X_r_test[i] @ best_beta_r[pred_map[i]] for i in range(len(pred_map))])

    Yi_hat = Yc_hat - Yr_hat
    Ys_hat = Yc_hat + Yr_hat

    # =========================
    # TEST METRICS
    # =========================
    CR, ER, RI = interval_metrics(Y_i_test, Y_s_test, Yi_hat, Ys_hat)
    r2L = r2_interval(Y_i_test, Yi_hat)
    r2U = r2_interval(Y_s_test, Ys_hat)
    r2LU = (r2L + r2U)/2

    r2C = r2_interval(Y_c_test, Yc_hat)
    r2R = r2_interval(Y_r_test, Yr_hat)
    r2CR = (r2C + r2R)/2

    rmseL = np.sqrt(mean_squared_error(Y_i_test, Yi_hat))
    rmseU = np.sqrt(mean_squared_error(Y_s_test, Ys_hat))
    rmseLU = np.sqrt(np.mean((Y_i_test-Yi_hat)**2 + (Y_s_test-Ys_hat)**2))

    rmseC = np.sqrt(mean_squared_error(Y_c_test, Yc_hat))
    rmseR = np.sqrt(mean_squared_error(Y_r_test, Yr_hat))
    rmseCR = np.sqrt(np.mean((Y_c_test-Yc_hat)**2 + (Y_r_test-Yr_hat)**2))

    # =========================
    #  TRAIN
    # =========================

    Yc_hat_tr = np.array([X_c_train[i] @ best_beta_c[pred_map_tr[i]] for i in range(len(pred_map_tr))])
    Yr_hat_tr = np.array([X_r_train[i] @ best_beta_r[pred_map_tr[i]] for i in range(len(pred_map_tr))])

    Yi_hat_tr = Yc_hat_tr - Yr_hat_tr
    Ys_hat_tr = Yc_hat_tr + Yr_hat_tr

    CRt, ERt, RIt = interval_metrics(Y_i_train, Y_s_train, Yi_hat_tr, Ys_hat_tr)
    r2Lt = r2_interval(Y_i_train, Yi_hat_tr)
    r2Ut = r2_interval(Y_s_train, Ys_hat_tr)
    r2LUt = (r2Lt + r2Ut)/2

    r2Ct = r2_interval(Y_c_train, Yc_hat_tr)
    r2Rt = r2_interval(Y_r_train, Yr_hat_tr)
    r2CRt = (r2Ct + r2Rt)/2

    rmseLt = np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr))
    rmseUt = np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr))
    rmseLUt = np.sqrt(np.mean((Y_i_train-Yi_hat_tr)**2 + (Y_s_train-Ys_hat_tr)**2))

    rmseCt = np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr))
    rmseRt = np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr))
    rmseCRt = np.sqrt(np.mean((Y_c_train-Yc_hat_tr)**2 + (Y_r_train-Yr_hat_tr)**2))

    # =========================
    #  CLUSTER
    # =========================
    ari_tr = adjusted_rand_score(true_labels_train, best_cluster)
    nmi_tr = normalized_mutual_info_score(true_labels_train, best_cluster)

    ari_test = adjusted_rand_score(true_labels_test, pred_map)
    nmi_test = normalized_mutual_info_score(true_labels_test, pred_map)

    # =========================
    #  SAVING
    # =========================

    # RMSE
    all_rmse_L_test.append(rmseL)
    all_rmse_U_test.append(rmseU)
    all_rmse_LU_test.append(rmseLU)

    all_rmse_L_train.append(rmseLt)
    all_rmse_U_train.append(rmseUt)
    all_rmse_LU_train.append(rmseLUt)

    all_rmse_c_test.append(rmseC)
    all_rmse_r_test.append(rmseR)
    all_rmse_conj_c_r_test.append(rmseCR)

    all_rmse_c_train.append(rmseCt)
    all_rmse_r_train.append(rmseRt)
    all_rmse_conj_c_r_train.append(rmseCRt)

    # Interval
    all_CR_test.append(CR)
    all_ER_test.append(ER)
    all_RI_test.append(RI)

    all_CR_train.append(CRt)
    all_ER_train.append(ERt)
    all_RI_train.append(RIt)

    # Correlation
    all_r2L_test.append(r2L)
    all_r2U_test.append(r2U)
    all_r2LU_test.append(r2LU)

    all_r2C_test.append(r2C)
    all_r2R_test.append(r2R)
    all_r2CR_test.append(r2CR)

    all_r2L_train.append(r2Lt)
    all_r2U_train.append(r2Ut)
    all_r2LU_train.append(r2LUt)

    all_r2C_train.append(r2Ct)
    all_r2R_train.append(r2Rt)
    all_r2CR_train.append(r2CRt)

    # Cluster
    all_ari_tr.append(ari_tr)
    all_nmi_tr.append(nmi_tr)

    all_ari_test.append(ari_test)
    all_nmi_test.append(nmi_test)

    print("\n" + "="*60)
    print(f"METRICS")
    print("="*60)

    # =========================
    #  INTERVAL
    # =========================
    print("\n INTERVAL")
    print("-"*40)

    print(f"CR  -> test: {CR:.4f} | train: {CRt:.4f}")
    print(f"ER  -> test: {ER:.4f} | train: {ERt:.4f}")
    print(f"RI  -> test: {RI:.4f} | train: {RIt:.4f}")

    # =========================
    #  CORRELATION (r²)
    # =========================
    print("\n CORRELAÇÃO (r²)")
    print("-"*40)

    print(f"r2_L  -> test: {r2L:.4f} | train: {r2Lt:.4f}")
    print(f"r2_U  -> test: {r2U:.4f} | train: {r2Ut:.4f}")
    print(f"r2_LU -> test: {r2LU:.4f} | train: {r2LUt:.4f}")

    print(f"r2_C  -> test: {r2C:.4f} | train: {r2Ct:.4f}")
    print(f"r2_R  -> test: {r2R:.4f} | train: {r2Rt:.4f}")
    print(f"r2_CR -> test: {r2CR:.4f} | train: {r2CRt:.4f}")

    # =========================
    #  RMSE
    # =========================
    print("\n RMSE")
    print("-"*40)

    print(f"RMSE_L  -> test: {rmseL:.4f} | train: {rmseLt:.4f}")
    print(f"RMSE_U  -> test: {rmseU:.4f} | train: {rmseUt:.4f}")
    print(f"RMSE_LU -> test: {rmseLU:.4f} | train: {rmseLUt:.4f}")

    print(f"RMSE_C  -> test: {rmseC:.4f} | train: {rmseCt:.4f}")
    print(f"RMSE_R  -> test: {rmseR:.4f} | train: {rmseRt:.4f}")
    print(f"RMSE_CR -> test: {rmseCR:.4f} | train: {rmseCRt:.4f}")

    # =========================
    #  CLUSTER
    # =========================
    print("\n CLUSTER")
    print("-"*40)

    print(f"ARI -> test: {ari_test:.4f} | train: {ari_tr:.4f}")
    print(f"NMI -> test: {nmi_test:.4f} | train: {nmi_tr:.4f}")

    print()

# =========================
#  Mean and Sd
# =========================
print("\n" + "="*100)
print("Results (Avg ± Sd) - MONTE CARLO")
print("="*100)

def mean_std(arr):
    return np.mean(arr), np.std(arr)

print("\n TEST")
print("-"*60)

def print_metric(name, arr):
    m, s = mean_std(arr)
    print(f"{name}: {m:.4f} ± {s:.4f}")

# RMSE
print_metric("RMSE_L", all_rmse_L_test)
print_metric("RMSE_U", all_rmse_U_test)
print_metric("RMSE_LU", all_rmse_LU_test)
print_metric("RMSE_C", all_rmse_c_test)
print_metric("RMSE_R", all_rmse_r_test)
print_metric("RMSE_CR", all_rmse_conj_c_r_test)

# Interval
print_metric("CR", all_CR_test)
print_metric("ER", all_ER_test)
print_metric("RI", all_RI_test)

# Correlation
print_metric("r2_L", all_r2L_test)
print_metric("r2_U", all_r2U_test)
print_metric("r2_LU", all_r2LU_test)
print_metric("r2_C", all_r2C_test)
print_metric("r2_R", all_r2R_test)
print_metric("r2_CR", all_r2CR_test)

print("\n TRAIN")
print("-"*60)

print_metric("RMSE_L", all_rmse_L_train)
print_metric("RMSE_U", all_rmse_U_train)
print_metric("RMSE_LU", all_rmse_LU_train)
print_metric("RMSE_C", all_rmse_c_train)
print_metric("RMSE_R", all_rmse_r_train)
print_metric("RMSE_CR", all_rmse_conj_c_r_train)

print_metric("CR", all_CR_train)
print_metric("ER", all_ER_train)
print_metric("RI", all_RI_train)

print_metric("r2_L", all_r2L_train)
print_metric("r2_U", all_r2U_train)
print_metric("r2_LU", all_r2LU_train)
print_metric("r2_C", all_r2C_train)
print_metric("r2_R", all_r2R_train)
print_metric("r2_CR", all_r2CR_train)

print("\n CLUSTER TRAIN")
print("-"*60)
print_metric("ARI", all_ari_tr)
print_metric("NMI", all_nmi_tr)

print("\n CLUSTER TEST")
print("-"*60)
print_metric("ARI", all_ari_test)
print_metric("NMI", all_nmi_test)

print()

### **$$\text{Overall }\boldsymbol{J}_{WCLRi} \text{ and Variants}$$**

In [ ]:
%%time
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import numpy as np

def interval_metrics(Y_i, Y_s, Y_i_hat, Y_s_hat):
    w_int = np.maximum(0, np.minimum(Y_s, Y_s_hat) - np.maximum(Y_i, Y_i_hat))
    w_union = np.maximum(Y_s, Y_s_hat) - np.minimum(Y_i, Y_i_hat)
    w_y = Y_s - Y_i
    w_yhat = Y_s_hat - Y_i_hat

    CR = np.mean(w_int / (w_y + 1e-12))
    ER = np.mean(w_int / (w_yhat + 1e-12))
    RI = np.mean(w_int / (w_union + 1e-12))

    return CR, ER, RI

def r2_interval(y, yhat):
    if np.std(y) == 0 or np.std(yhat) == 0:
        return 0
    cov = np.cov(y, yhat, bias=True)[0,1]
    return (cov / (np.std(y) * np.std(yhat)))**2

# =========================
# PARAMETERS
# =========================

K_Values = range(3,4)
epsilon = 1e-6
r = 100  # RANDOM INICIALIZATION
knn_Values = range(1,18)
alpha_Values = [1e-5, 1e-2, 1e0, 1e2, 1e5]
n_mc = 50

# =========================
# RESULT LIST
# =========================

# RMSE
all_rmse_L_test, all_rmse_U_test, all_rmse_LU_test = [], [], []
all_rmse_L_train, all_rmse_U_train, all_rmse_LU_train = [], [], []

all_rmse_c_test, all_rmse_r_test, all_rmse_conj_c_r_test = [], [], []
all_rmse_c_train, all_rmse_r_train, all_rmse_conj_c_r_train = [], [], []

# Interval
all_CR_test, all_ER_test, all_RI_test = [], [], []
all_CR_train, all_ER_train, all_RI_train = [], [], []

# Correlation
all_r2L_test, all_r2U_test, all_r2LU_test = [], [], []
all_r2C_test, all_r2R_test, all_r2CR_test = [], [], []

all_r2L_train, all_r2U_train, all_r2LU_train = [], [], []
all_r2C_train, all_r2R_train, all_r2CR_train = [], [], []

# Cluster
all_ari_test, all_nmi_test, all_ari_tr, all_nmi_tr = [], [], [], []

# =========================
# MONTE CARLO
# =========================
for mc in range(n_mc):
    print(f"\n Monte Carlo {mc+1}/{n_mc}")

    df = generate_dataset1(n_per_class=100, seed=(mc+10))
    true_labels = df['Classe'].values

    X_i0 = df[['X1_min', 'X2_min']].values
    X_s0 = df[['X1_max', 'X2_max']].values
    Y_i = df['Y_min'].values
    Y_s = df['Y_max'].values

    X_c0 = df[['X1_c', 'X2_c']].values
    X_r0 = df[['X1_r', 'X2_r']].values
    Y_c = df['Y_c'].values
    Y_r = df['Y_r'].values

    n = len(df)
    intercept = np.ones((n,1))

    X_i = np.hstack((intercept, X_i0))
    X_s = np.hstack((intercept, X_s0))
    X_c = np.hstack((intercept, X_c0))
    X_r = np.hstack((intercept, X_r0))

    # =========================
    #  SPLIT
    # =========================
    X_c_train, X_c_test, X_r_train, X_r_test, \
    X_i_train, X_i_test, X_s_train, X_s_test, \
    Y_c_train, Y_c_test, Y_r_train, Y_r_test, \
    Y_i_train, Y_i_test, Y_s_train, Y_s_test, \
    true_labels_train, true_labels_test = train_test_split(
        X_c, X_r, X_i, X_s,
        Y_c, Y_r, Y_i, Y_s,
        true_labels,

        test_size=0.2,

        stratify=true_labels,

        random_state=mc
    )

    # =========================
    #  K-FOLD
    # =========================
    best_K, best_knn, best_alpha = kfold_cv_with_knn_V(
        X_c_train, X_r_train, Y_c_train, Y_r_train,
        X_i_train, X_s_train, Y_i_train, Y_s_train,
        true_labels_train, K_Values, knn_Values, alpha_Values, r, epsilon
    )

    # =========================
    #  FINAL TRAIN
    # =========================
    best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments_V(
        X_c_train, X_r_train, Y_c_train, Y_r_train,
        X_i_train, X_s_train, Y_i_train, Y_s_train,
        best_K, best_alpha, r, epsilon
    )

    print("\n" + "="*80)
    print(f"RESULTS - EXPERIMENT {mc+1}")
    print("="*80)

    for k in range(len(best_beta_c)):
        n_k = np.sum(best_cluster == k)
        print(f"Group {k} (n={n_k})")
        print(f"  beta_c: {np.round(best_beta_c[k], 4)}")
        print(f"  beta_r: {np.round(best_beta_r[k], 4)}")
        print()

    # =========================
    #  PEDICTION
    # =========================
    if best_K == 1:
        best_knn = None
        pred_map = np.zeros(len(X_i_test), dtype=int)
        pred_map_tr = np.zeros(len(X_i_train), dtype=int)

    else:
        X_train_combined = np.hstack((X_c_train[:, 1:], X_r_train[:, 1:]))
        X_test_combined   = np.hstack((X_c_test[:, 1:],   X_r_test[:, 1:]))

        knn = KNeighborsClassifier(
            n_neighbors=best_knn,
            metric='euclidean'
        )

        knn.fit(X_train_combined, best_cluster)

        # =========================
        #  TEST
        # =========================
        pred_groups = knn.predict(X_test_combined)

        pred_map = pred_groups

        # =========================
        #  TRAIN
        # =========================
        pred_map_tr = best_cluster

    Yc_hat = np.array([X_c_test[i] @ best_beta_c[pred_map[i]] for i in range(len(pred_map))])
    Yr_hat = np.array([X_r_test[i] @ best_beta_r[pred_map[i]] for i in range(len(pred_map))])

    Yi_hat = Yc_hat - Yr_hat
    Ys_hat = Yc_hat + Yr_hat

    # =========================
    # TEST METRICS
    # =========================
    CR, ER, RI = interval_metrics(Y_i_test, Y_s_test, Yi_hat, Ys_hat)
    r2L = r2_interval(Y_i_test, Yi_hat)
    r2U = r2_interval(Y_s_test, Ys_hat)
    r2LU = (r2L + r2U)/2

    r2C = r2_interval(Y_c_test, Yc_hat)
    r2R = r2_interval(Y_r_test, Yr_hat)
    r2CR = (r2C + r2R)/2

    rmseL = np.sqrt(mean_squared_error(Y_i_test, Yi_hat))
    rmseU = np.sqrt(mean_squared_error(Y_s_test, Ys_hat))
    rmseLU = np.sqrt(np.mean((Y_i_test-Yi_hat)**2 + (Y_s_test-Ys_hat)**2))

    rmseC = np.sqrt(mean_squared_error(Y_c_test, Yc_hat))
    rmseR = np.sqrt(mean_squared_error(Y_r_test, Yr_hat))
    rmseCR = np.sqrt(np.mean((Y_c_test-Yc_hat)**2 + (Y_r_test-Yr_hat)**2))

    # =========================
    #  TRAIN
    # =========================

    Yc_hat_tr = np.array([X_c_train[i] @ best_beta_c[pred_map_tr[i]] for i in range(len(pred_map_tr))])
    Yr_hat_tr = np.array([X_r_train[i] @ best_beta_r[pred_map_tr[i]] for i in range(len(pred_map_tr))])

    Yi_hat_tr = Yc_hat_tr - Yr_hat_tr
    Ys_hat_tr = Yc_hat_tr + Yr_hat_tr

    CRt, ERt, RIt = interval_metrics(Y_i_train, Y_s_train, Yi_hat_tr, Ys_hat_tr)
    r2Lt = r2_interval(Y_i_train, Yi_hat_tr)
    r2Ut = r2_interval(Y_s_train, Ys_hat_tr)
    r2LUt = (r2Lt + r2Ut)/2

    r2Ct = r2_interval(Y_c_train, Yc_hat_tr)
    r2Rt = r2_interval(Y_r_train, Yr_hat_tr)
    r2CRt = (r2Ct + r2Rt)/2

    rmseLt = np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr))
    rmseUt = np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr))
    rmseLUt = np.sqrt(np.mean((Y_i_train-Yi_hat_tr)**2 + (Y_s_train-Ys_hat_tr)**2))

    rmseCt = np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr))
    rmseRt = np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr))
    rmseCRt = np.sqrt(np.mean((Y_c_train-Yc_hat_tr)**2 + (Y_r_train-Yr_hat_tr)**2))

    # =========================
    #  CLUSTER
    # =========================
    ari_tr = adjusted_rand_score(true_labels_train, best_cluster)
    nmi_tr = normalized_mutual_info_score(true_labels_train, best_cluster)

    ari_test = adjusted_rand_score(true_labels_test, pred_map)
    nmi_test = normalized_mutual_info_score(true_labels_test, pred_map)

    # =========================
    #  SAVING
    # =========================

    # RMSE
    all_rmse_L_test.append(rmseL)
    all_rmse_U_test.append(rmseU)
    all_rmse_LU_test.append(rmseLU)

    all_rmse_L_train.append(rmseLt)
    all_rmse_U_train.append(rmseUt)
    all_rmse_LU_train.append(rmseLUt)

    all_rmse_c_test.append(rmseC)
    all_rmse_r_test.append(rmseR)
    all_rmse_conj_c_r_test.append(rmseCR)

    all_rmse_c_train.append(rmseCt)
    all_rmse_r_train.append(rmseRt)
    all_rmse_conj_c_r_train.append(rmseCRt)

    # Interval
    all_CR_test.append(CR)
    all_ER_test.append(ER)
    all_RI_test.append(RI)

    all_CR_train.append(CRt)
    all_ER_train.append(ERt)
    all_RI_train.append(RIt)

    # Correlation
    all_r2L_test.append(r2L)
    all_r2U_test.append(r2U)
    all_r2LU_test.append(r2LU)

    all_r2C_test.append(r2C)
    all_r2R_test.append(r2R)
    all_r2CR_test.append(r2CR)

    all_r2L_train.append(r2Lt)
    all_r2U_train.append(r2Ut)
    all_r2LU_train.append(r2LUt)

    all_r2C_train.append(r2Ct)
    all_r2R_train.append(r2Rt)
    all_r2CR_train.append(r2CRt)

    # Cluster
    all_ari_tr.append(ari_tr)
    all_nmi_tr.append(nmi_tr)

    all_ari_test.append(ari_test)
    all_nmi_test.append(nmi_test)

    print("\n" + "="*60)
    print(f" METRICS")
    print("="*60)

    # =========================
    #  INTERVAL
    # =========================
    print("\n INTERVAL")
    print("-"*40)

    print(f"CR  -> test: {CR:.4f} | train: {CRt:.4f}")
    print(f"ER  -> test: {ER:.4f} | train: {ERt:.4f}")
    print(f"RI  -> test: {RI:.4f} | train: {RIt:.4f}")

    # =========================
    #  CORRELATION (r²)
    # =========================
    print("\n CORRELAÇÃO (r²)")
    print("-"*40)

    print(f"r2_L  -> test: {r2L:.4f} | train: {r2Lt:.4f}")
    print(f"r2_U  -> test: {r2U:.4f} | train: {r2Ut:.4f}")
    print(f"r2_LU -> test: {r2LU:.4f} | train: {r2LUt:.4f}")

    print(f"r2_C  -> test: {r2C:.4f} | train: {r2Ct:.4f}")
    print(f"r2_R  -> test: {r2R:.4f} | train: {r2Rt:.4f}")
    print(f"r2_CR -> test: {r2CR:.4f} | train: {r2CRt:.4f}")

    # =========================
    #  RMSE
    # =========================
    print("\n RMSE")
    print("-"*40)

    print(f"RMSE_L  -> test: {rmseL:.4f} | train: {rmseLt:.4f}")
    print(f"RMSE_U  -> test: {rmseU:.4f} | train: {rmseUt:.4f}")
    print(f"RMSE_LU -> test: {rmseLU:.4f} | train: {rmseLUt:.4f}")

    print(f"RMSE_C  -> test: {rmseC:.4f} | train: {rmseCt:.4f}")
    print(f"RMSE_R  -> test: {rmseR:.4f} | train: {rmseRt:.4f}")
    print(f"RMSE_CR -> test: {rmseCR:.4f} | train: {rmseCRt:.4f}")

    # =========================
    #  CLUSTER
    # =========================
    print("\n CLUSTER")
    print("-"*40)

    print(f"ARI -> test: {ari_test:.4f} | train: {ari_tr:.4f}")
    print(f"NMI -> test: {nmi_test:.4f} | train: {nmi_tr:.4f}")

    print()

# =========================
#  Mean and Sd
# =========================
print("\n" + "="*100)
print("Results (Avg ± Sd) - MONTE CARLO")
print("="*100)

def mean_std(arr):
    return np.mean(arr), np.std(arr)

print("\n TEST")
print("-"*60)

def print_metric(name, arr):
    m, s = mean_std(arr)
    print(f"{name}: {m:.4f} ± {s:.4f}")

# RMSE
print_metric("RMSE_L", all_rmse_L_test)
print_metric("RMSE_U", all_rmse_U_test)
print_metric("RMSE_LU", all_rmse_LU_test)
print_metric("RMSE_C", all_rmse_c_test)
print_metric("RMSE_R", all_rmse_r_test)
print_metric("RMSE_CR", all_rmse_conj_c_r_test)

# Intervalo
print_metric("CR", all_CR_test)
print_metric("ER", all_ER_test)
print_metric("RI", all_RI_test)

# Correlação
print_metric("r2_L", all_r2L_test)
print_metric("r2_U", all_r2U_test)
print_metric("r2_LU", all_r2LU_test)
print_metric("r2_C", all_r2C_test)
print_metric("r2_R", all_r2R_test)
print_metric("r2_CR", all_r2CR_test)

print("\n TRAIN")
print("-"*60)

print_metric("RMSE_L", all_rmse_L_train)
print_metric("RMSE_U", all_rmse_U_train)
print_metric("RMSE_LU", all_rmse_LU_train)
print_metric("RMSE_C", all_rmse_c_train)
print_metric("RMSE_R", all_rmse_r_train)
print_metric("RMSE_CR", all_rmse_conj_c_r_train)

print_metric("CR", all_CR_train)
print_metric("ER", all_ER_train)
print_metric("RI", all_RI_train)

print_metric("r2_L", all_r2L_train)
print_metric("r2_U", all_r2U_train)
print_metric("r2_LU", all_r2LU_train)
print_metric("r2_C", all_r2C_train)
print_metric("r2_R", all_r2R_train)
print_metric("r2_CR", all_r2CR_train)

print("\n CLUSTER TRAIN")
print("-"*60)
print_metric("ARI", all_ari_tr)
print_metric("NMI", all_nmi_tr)

print("\n CLUSTER TEST")
print("-"*60)
print_metric("ARI", all_ari_test)
print_metric("NMI", all_nmi_test)

print()

## **Bootstrap .632+**

### **$$\text{Overall }\boldsymbol{J}_{CLRi}$$**

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

def r2_interval(y, yhat):

    if np.std(y) == 0 or np.std(yhat) == 0:
        return 0

    cov = np.cov(y, yhat, bias=True)[0,1]

    return (cov / (np.std(y) * np.std(yhat)))**2


def interval_metrics(Y_i, Y_s, Y_i_hat, Y_s_hat):

    w_int = np.maximum(
        0,
        np.minimum(Y_s, Y_s_hat) -
        np.maximum(Y_i, Y_i_hat)
    )

    w_union = (
        np.maximum(Y_s, Y_s_hat) -
        np.minimum(Y_i, Y_i_hat)
    )

    w_y = Y_s - Y_i
    w_yhat = Y_s_hat - Y_i_hat

    CR = np.mean(w_int / (w_y + 1e-12))

    ER = np.mean(w_int / (w_yhat + 1e-12))

    RI = np.mean(w_int / (w_union + 1e-12))

    return CR, ER, RI


# =========================================================
#  BOOTSTRAP 0.632+ (ERROR)
# =========================================================

def bootstrap_632_plus_error(err_train,
                             err_oob,
                             err_noinfo):

    R = (err_oob - err_train) / (
        (err_noinfo - err_train) + 1e-12
    )

    R = np.clip(R, 0, 1)

    w = 0.632 / (1 - 0.368 * R)

    return (
        (1 - w)*err_train +
        w*err_oob
    )


# =========================================================
#  BOOTSTRAP 0.632+ (SCORE)
# =========================================================

def bootstrap_632_plus_score(score_train,
                             score_oob,
                             score_noinfo):

    R = (score_train - score_oob) / (
        (score_train - score_noinfo) + 1e-12
    )

    R = np.clip(R, 0, 1)

    w = 0.632 / (1 - 0.368 * R)

    return (
        (1 - w)*score_train +
        w*score_oob
    )


# =========================================================
#  PARAMETERS
# =========================================================
K_values = range(1,4)
epsilon = 1e-6
r = 100  # RANDOM INICIALIZATION
knn_values = range(1,6)
n_bootstrap = 100

metric_names = [

    "RMSE_L",
    "RMSE_U",
    "RMSE_LU",

    "RMSE_C",
    "RMSE_R",
    "RMSE_CR",

    "CR",
    "ER",
    "RI",

    "r2_L",
    "r2_U",
    "r2_LU",

    "r2_C",
    "r2_R",
    "r2_CR",

    "R2_1_L",
    "R2_1_U",
    "R2_1_LU",

    "R2_1_C",
    "R2_1_R",
    "R2_1_CR",

    "R2_2_L",
    "R2_2_U",
    "R2_2_LU",

    "R2_2_C",
    "R2_2_R",
    "R2_2_CR"
]

all_train = []
all_oob = []

all_632 = []
all_632plus = []

# =========================================================
#  LOOP BOOTSTRAP
# =========================================================

for b in range(n_bootstrap):

    print(f"\nBootstrap {b+1}/{n_bootstrap}")

    # Gerar base
    df = dataset(df0)

    X_i0 = df[['X1_min', 'X2_min']].values
    X_s0 = df[['X1_max', 'X2_max']].values
    Y_i = df['Y_min'].values
    Y_s = df['Y_max'].values

    X_c0 = df[['X1_c', 'X2_c']].values
    X_r0 = df[['X1_r', 'X2_r']].values
    Y_c = df['Y_c'].values
    Y_r = df['Y_r'].values

    n = len(df)
    intercept = np.ones((n, 1))

    X_i = np.hstack((intercept, X_i0))
    X_s = np.hstack((intercept, X_s0))
    X_c = np.hstack((intercept, X_c0))
    X_r = np.hstack((intercept, X_r0))

    idx_boot = np.random.choice(
        n,
        size=n,
        replace=True
    )

    idx_oob = np.setdiff1d(
        np.arange(n),
        idx_boot
    )

    if len(idx_oob) == 0:
        continue

    # =====================================================
    #  SPLIT
    # =====================================================

    X_c_train = X_c[idx_boot]
    X_r_train = X_r[idx_boot]

    X_i_train = X_i[idx_boot]
    X_s_train = X_s[idx_boot]

    Y_c_train = Y_c[idx_boot]
    Y_r_train = Y_r[idx_boot]

    Y_i_train = Y_i[idx_boot]
    Y_s_train = Y_s[idx_boot]

    X_c_test = X_c[idx_oob]
    X_r_test = X_r[idx_oob]

    X_i_test = X_i[idx_oob]
    X_s_test = X_s[idx_oob]

    Y_c_test = Y_c[idx_oob]
    Y_r_test = Y_r[idx_oob]

    Y_i_test = Y_i[idx_oob]
    Y_s_test = Y_s[idx_oob]

    # =====================================================
    #  TUNING
    # =====================================================

    best_K, best_knn = bootstrap_cv_with_knn(

        X_c_train,
        X_r_train,

        Y_c_train,
        Y_r_train,

        X_i_train,
        X_s_train,

        Y_i_train,
        Y_s_train,

        K_values,
        knn_values,

        r,
        epsilon,

        B=100
    )

    # =====================================================
    #  FINAL TRAIN
    # =====================================================

    best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments(

        X_c_train,
        X_r_train,

        Y_c_train,
        Y_r_train,

        X_i_train,
        X_s_train,

        Y_i_train,
        Y_s_train,

        best_K,

        r,
        epsilon
    )

    print("\n" + "="*80)
    print(f" RESULTS - EXPERIMENT {n_bootstrap+1}")
    print("="*80)

    for k in range(len(best_beta_c)):
        n_k = np.sum(best_cluster == k)
        print(f"Group {k} (n={n_k})")
        print(f"  beta_c: {np.round(best_beta_c[k], 4)}")
        print(f"  beta_r: {np.round(best_beta_r[k], 4)}")
        print()

    # =====================================================
    #  KNN
    # =====================================================

    if best_K == 1:

        pred_map = np.zeros(len(X_i_test), dtype=int)

        pred_map_tr = np.zeros(len(X_i_train), dtype=int)

        best_knn = None

    else:

        X_train_combined = np.hstack((
            X_i_train[:,1:],
            X_s_train[:,1:]
        ))

        X_test_combined = np.hstack((
            X_i_test[:,1:],
            X_s_test[:,1:]
        ))

        knn = KNeighborsClassifier(
            n_neighbors=best_knn,
            metric='euclidean'
        )

        knn.fit(
            X_train_combined,
            best_cluster
        )

        pred_map = knn.predict(
            X_test_combined
        )

        pred_map_tr = knn.predict(
            X_train_combined
        )

    # =====================================================
    #  PREDICT
    # =====================================================

    def predict(Xc, Xr, mapping):

        Yc_hat = np.array([
            Xc[i] @ best_beta_c[mapping[i]]
            for i in range(len(mapping))
        ])

        Yr_hat = np.array([
            Xr[i] @ best_beta_r[mapping[i]]
            for i in range(len(mapping))
        ])

        Yi_hat = Yc_hat - Yr_hat
        Ys_hat = Yc_hat + Yr_hat

        return Yc_hat, Yr_hat, Yi_hat, Ys_hat

    Yc_hat, Yr_hat, Yi_hat, Ys_hat = predict(
        X_c_test,
        X_r_test,
        pred_map
    )

    Yc_hat_tr, Yr_hat_tr, Yi_hat_tr, Ys_hat_tr = predict(
        X_c_train,
        X_r_train,
        pred_map_tr
    )

    # =====================================================
    #  METRICS TRAIN
    # =====================================================

    CRt, ERt, RIt = interval_metrics(
        Y_i_train,
        Y_s_train,
        Yi_hat_tr,
        Ys_hat_tr
    )

    CR, ER, RI = interval_metrics(
        Y_i_test,
        Y_s_test,
        Yi_hat,
        Ys_hat
    )

    train_metrics = np.array([

        # RMSE
        np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr)),
        np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr)),

        np.sqrt(np.mean(
            (Y_i_train - Yi_hat_tr)**2 +
            (Y_s_train - Ys_hat_tr)**2
        )),

        np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr)),
        np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr)),

        np.sqrt(np.mean(
            (Y_c_train - Yc_hat_tr)**2 +
            (Y_r_train - Yr_hat_tr)**2
        )),

        # INTERVAL
        CRt,
        ERt,
        RIt,

        # r2
        r2_interval(Y_i_train, Yi_hat_tr),
        r2_interval(Y_s_train, Ys_hat_tr),

        (
            r2_interval(Y_i_train, Yi_hat_tr) +
            r2_interval(Y_s_train, Ys_hat_tr)
        ) / 2,

        r2_interval(Y_c_train, Yc_hat_tr),
        r2_interval(Y_r_train, Yr_hat_tr),

        (
            r2_interval(Y_c_train, Yc_hat_tr) +
            r2_interval(Y_r_train, Yr_hat_tr)
        ) / 2

    ])

    # =====================================================
    #  METRICS OOB
    # =====================================================

    oob_metrics = np.array([

        # RMSE
        np.sqrt(mean_squared_error(Y_i_test, Yi_hat)),
        np.sqrt(mean_squared_error(Y_s_test, Ys_hat)),

        np.sqrt(np.mean(
            (Y_i_test - Yi_hat)**2 +
            (Y_s_test - Ys_hat)**2
        )),

        np.sqrt(mean_squared_error(Y_c_test, Yc_hat)),
        np.sqrt(mean_squared_error(Y_r_test, Yr_hat)),

        np.sqrt(np.mean(
            (Y_c_test - Yc_hat)**2 +
            (Y_r_test - Yr_hat)**2
        )),

        # Interval
        CR,
        ER,
        RI,

        # r2
        r2_interval(Y_i_test, Yi_hat),
        r2_interval(Y_s_test, Ys_hat),

        (
            r2_interval(Y_i_test, Yi_hat) +
            r2_interval(Y_s_test, Ys_hat)
        ) / 2,

        r2_interval(Y_c_test, Yc_hat),
        r2_interval(Y_r_test, Yr_hat),

        (
            r2_interval(Y_c_test, Yc_hat) +
            r2_interval(Y_r_test, Yr_hat)
        ) / 2
    ])

################################################################
    # =====================================================
    #  NO-INFORMATION BASELINE
    # =====================================================

    # -----------------------------------------------------
    # LOWER E UPPER
    # -----------------------------------------------------

    baseline_L = np.mean(Y_i_train)

    baseline_U = np.mean(Y_s_train)

    # -----------------------------------------------------
    # CENTRE E RANGE
    # -----------------------------------------------------

    baseline_C = np.mean(Y_c_train)

    baseline_R = np.mean(Y_r_train)

    # -----------------------------------------------------
    # PREDICT BASELINE
    # -----------------------------------------------------

    Yi_baseline = np.full(
        len(Y_i_test),
        baseline_L
    )

    Ys_baseline = np.full(
        len(Y_s_test),
        baseline_U
    )

    Yc_baseline = np.full(
        len(Y_c_test),
        baseline_C
    )

    Yr_baseline = np.full(
        len(Y_r_test),
        baseline_R
    )

    # =====================================================
    #  METRICS NO-INFORMATION
    # =====================================================

    CR_base, ER_base, RI_base = interval_metrics(

        Y_i_test,
        Y_s_test,

        Yi_baseline,
        Ys_baseline
    )

    noinfo_metrics = np.array([

        # =================================================
        # RMSE
        # =================================================

        np.sqrt(mean_squared_error(
            Y_i_test,
            Yi_baseline
        )),

        np.sqrt(mean_squared_error(
            Y_s_test,
            Ys_baseline
        )),

        np.sqrt(np.mean(
            (Y_i_test - Yi_baseline)**2 +
            (Y_s_test - Ys_baseline)**2
        )),

        np.sqrt(mean_squared_error(
            Y_c_test,
            Yc_baseline
        )),

        np.sqrt(mean_squared_error(
            Y_r_test,
            Yr_baseline
        )),

        np.sqrt(np.mean(
            (Y_c_test - Yc_baseline)**2 +
            (Y_r_test - Yr_baseline)**2
        )),

        # =================================================
        # INTERVAL
        # =================================================

        CR_base,
        ER_base,
        RI_base,

        # =================================================
        # r2
        # =================================================

        r2_interval(Y_i_test, Yi_baseline),

        r2_interval(Y_s_test, Ys_baseline),

        (
            r2_interval(Y_i_test, Yi_baseline) +
            r2_interval(Y_s_test, Ys_baseline)
        ) / 2,

        r2_interval(Y_c_test, Yc_baseline),

        r2_interval(Y_r_test, Yr_baseline),

        (
            r2_interval(Y_c_test, Yc_baseline) +
            r2_interval(Y_r_test, Yr_baseline)
        ) / 2
    ])

################################################################

    # =====================================================
    #  0.632
    # =====================================================

    metrics632 = (
        0.368 * train_metrics +
        0.632 * oob_metrics
    )

    # =====================================================
    #  0.632+
    # =====================================================


    metrics632plus = []

    for i in range(len(train_metrics)):

        # =====================================================
        #  METRICS ERROR
        # =====================================================

        if metric_names[i].startswith("RMSE"):

            val = bootstrap_632_plus_error(

                train_metrics[i],
                oob_metrics[i],
                noinfo_metrics[i]
            )

        # =====================================================
        #  METRICS SCORE
        # =====================================================

        else:

            val = bootstrap_632_plus_score(

                train_metrics[i],
                oob_metrics[i],
                noinfo_metrics[i]
            )

        metrics632plus.append(val)

    metrics632plus = np.array(metrics632plus)

    # =====================================================
    #  SAVING
    # =====================================================

    all_train.append(train_metrics)

    all_oob.append(oob_metrics)

    all_632.append(metrics632)

    all_632plus.append(metrics632plus)

# =========================
#  MEAN (SD)
# =========================
print("\n" + "="*100)
print(" RESULTS (AVG ± SD) - MONTE CARLO")
print("="*100)

all_train = np.array(all_train)

all_oob = np.array(all_oob)

all_632 = np.array(all_632)

all_632plus = np.array(all_632plus)

grupos = {

    "TRAIN": all_train,

    "BOOTSTRAP/OOB": all_oob,

    "BOOTSTRAP .632": all_632,

    "BOOTSTRAP .632+": all_632plus
}

for grupo, valores in grupos.items():

    print("\n" + "="*80)

    print(grupo)

    print("="*80)

    medias = np.mean(valores, axis=0)

    desvios = np.std(valores, axis=0)

    for i, nome in enumerate(metric_names):

        print(
            f"{nome}: "
            f"{medias[i]:.4f} ± {desvios[i]:.4f}"
        )

### **$$\text{Overall }\boldsymbol{J}_{WCLRi} \text{ and Variants}$$**

In [ ]:
%%time
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

def r2_interval(y, yhat):

    if np.std(y) == 0 or np.std(yhat) == 0:
        return 0

    cov = np.cov(y, yhat, bias=True)[0,1]

    return (cov / (np.std(y) * np.std(yhat)))**2


def interval_metrics(Y_i, Y_s, Y_i_hat, Y_s_hat):

    w_int = np.maximum(
        0,
        np.minimum(Y_s, Y_s_hat) -
        np.maximum(Y_i, Y_i_hat)
    )

    w_union = (
        np.maximum(Y_s, Y_s_hat) -
        np.minimum(Y_i, Y_i_hat)
    )

    w_y = Y_s - Y_i
    w_yhat = Y_s_hat - Y_i_hat

    CR = np.mean(w_int / (w_y + 1e-12))

    ER = np.mean(w_int / (w_yhat + 1e-12))

    RI = np.mean(w_int / (w_union + 1e-12))

    return CR, ER, RI


# =========================================================
#  BOOTSTRAP 0.632+ (ERROR)
# =========================================================

def bootstrap_632_plus_error(err_train,
                             err_oob,
                             err_noinfo):

    R = (err_oob - err_train) / (
        (err_noinfo - err_train) + 1e-12
    )

    R = np.clip(R, 0, 1)

    w = 0.632 / (1 - 0.368 * R)

    return (
        (1 - w)*err_train +
        w*err_oob
    )


# =========================================================
#  BOOTSTRAP 0.632+ (SCORE)
# =========================================================

def bootstrap_632_plus_score(score_train,
                             score_oob,
                             score_noinfo):

    R = (score_train - score_oob) / (
        (score_train - score_noinfo) + 1e-12
    )

    R = np.clip(R, 0, 1)

    w = 0.632 / (1 - 0.368 * R)

    return (
        (1 - w)*score_train +
        w*score_oob
    )


# =========================================================
#  PARAMETERS
# =========================================================
K_values = range(1,4)
epsilon = 1e-6
r = 100  # RANDOM INICIALIZATION
knn_values = range(1,6)
alpha_values = [1e-5, 1e-2, 1e0, 1e2, 1e5]
n_bootstrap = 100

metric_names = [

    "RMSE_L",
    "RMSE_U",
    "RMSE_LU",

    "RMSE_C",
    "RMSE_R",
    "RMSE_CR",

    "CR",
    "ER",
    "RI",

    "r2_L",
    "r2_U",
    "r2_LU",

    "r2_C",
    "r2_R",
    "r2_CR",

    "R2_1_L",
    "R2_1_U",
    "R2_1_LU",

    "R2_1_C",
    "R2_1_R",
    "R2_1_CR",

    "R2_2_L",
    "R2_2_U",
    "R2_2_LU",

    "R2_2_C",
    "R2_2_R",
    "R2_2_CR"
]

all_train = []
all_oob = []

all_632 = []
all_632plus = []

# =========================================================
#  LOOP BOOTSTRAP
# =========================================================

for b in range(n_bootstrap):

    print(f"\nBootstrap {b+1}/{n_bootstrap}")

    # Gerar base
    df = dataset(df0)

    X_i0 = df[['X1_min', 'X2_min']].values
    X_s0 = df[['X1_max', 'X2_max']].values
    Y_i = df['Y_min'].values
    Y_s = df['Y_max'].values

    X_c0 = df[['X1_c', 'X2_c']].values
    X_r0 = df[['X1_r', 'X2_r']].values
    Y_c = df['Y_c'].values
    Y_r = df['Y_r'].values

    n = len(df)
    intercept = np.ones((n, 1))

    X_i = np.hstack((intercept, X_i0))
    X_s = np.hstack((intercept, X_s0))
    X_c = np.hstack((intercept, X_c0))
    X_r = np.hstack((intercept, X_r0))

    idx_boot = np.random.choice(
        n,
        size=n,
        replace=True
    )

    idx_oob = np.setdiff1d(
        np.arange(n),
        idx_boot
    )

    if len(idx_oob) == 0:
        continue

    # =====================================================
    #  SPLIT
    # =====================================================

    X_c_train = X_c[idx_boot]
    X_r_train = X_r[idx_boot]

    X_i_train = X_i[idx_boot]
    X_s_train = X_s[idx_boot]

    Y_c_train = Y_c[idx_boot]
    Y_r_train = Y_r[idx_boot]

    Y_i_train = Y_i[idx_boot]
    Y_s_train = Y_s[idx_boot]

    X_c_test = X_c[idx_oob]
    X_r_test = X_r[idx_oob]

    X_i_test = X_i[idx_oob]
    X_s_test = X_s[idx_oob]

    Y_c_test = Y_c[idx_oob]
    Y_r_test = Y_r[idx_oob]

    Y_i_test = Y_i[idx_oob]
    Y_s_test = Y_s[idx_oob]

    # =====================================================
    #  TUNING
    # =====================================================

    best_K, best_knn, best_alpha = bootstrap_cv_with_knn_v(

        X_c_train,
        X_r_train,

        Y_c_train,
        Y_r_train,

        X_i_train,
        X_s_train,

        Y_i_train,
        Y_s_train,

        K_values,
        knn_values,
        alpha_values,

        r,
        epsilon,

        B=100
    )



    # =====================================================
    #  FINAL TRAIN
    # =====================================================

    best_J, best_beta_c, best_beta_r, best_cluster, best_V = run_multiple_experiments_V(

        X_c_train,
        X_r_train,

        Y_c_train,
        Y_r_train,

        X_i_train,
        X_s_train,

        Y_i_train,
        Y_s_train,

        best_K,
        best_alpha,

        r,
        epsilon
    )

    print("\n" + "="*80)
    print(f" RESULTS - EXPERIMENT {n_bootstrap+1}")
    print("="*80)

    for k in range(len(best_beta_c)):
        n_k = np.sum(best_cluster == k)
        print(f"Group {k} (n={n_k})")
        print(f"  beta_c: {np.round(best_beta_c[k], 4)}")
        print(f"  beta_r: {np.round(best_beta_r[k], 4)}")
        print()

    # =====================================================
    #  KNN
    # =====================================================

    if best_K == 1:

        pred_map = np.zeros(len(X_i_test), dtype=int)

        pred_map_tr = np.zeros(len(X_i_train), dtype=int)

        best_knn = None

    else:

        from sklearn.preprocessing import StandardScaler

        scaler = StandardScaler()

        X_train_combined0 = np.hstack((
            X_i_train[:,1:],
            X_s_train[:,1:]
        ))

        X_test_combined0 = np.hstack((
            X_i_test[:,1:],
            X_s_test[:,1:]
        ))

        X_train_combined = scaler.fit_transform(X_train_combined0)

        X_test_combined = scaler.transform(X_test_combined0)
        knn = KNeighborsClassifier(
            n_neighbors=best_knn,
            metric='euclidean'
        )

        knn.fit(
            X_train_combined,
            best_cluster
        )

        pred_map = knn.predict(
            X_test_combined
        )

        pred_map_tr = knn.predict(
            X_train_combined
        )

    # =====================================================
    #  PREDICT
    # =====================================================

    def predict(Xc, Xr, mapping):

        Yc_hat = np.array([
            Xc[i] @ best_beta_c[mapping[i]]
            for i in range(len(mapping))
        ])

        Yr_hat = np.array([
            Xr[i] @ best_beta_r[mapping[i]]
            for i in range(len(mapping))
        ])

        Yi_hat = Yc_hat - Yr_hat
        Ys_hat = Yc_hat + Yr_hat

        return Yc_hat, Yr_hat, Yi_hat, Ys_hat

    Yc_hat, Yr_hat, Yi_hat, Ys_hat = predict(
        X_c_test,
        X_r_test,
        pred_map
    )

    Yc_hat_tr, Yr_hat_tr, Yi_hat_tr, Ys_hat_tr = predict(
        X_c_train,
        X_r_train,
        pred_map_tr
    )

    # =====================================================
    #  METRICS TRAIN
    # =====================================================

    CRt, ERt, RIt = interval_metrics(
        Y_i_train,
        Y_s_train,
        Yi_hat_tr,
        Ys_hat_tr
    )

    CR, ER, RI = interval_metrics(
        Y_i_test,
        Y_s_test,
        Yi_hat,
        Ys_hat
    )

    train_metrics = np.array([

        # RMSE
        np.sqrt(mean_squared_error(Y_i_train, Yi_hat_tr)),
        np.sqrt(mean_squared_error(Y_s_train, Ys_hat_tr)),

        np.sqrt(np.mean(
            (Y_i_train - Yi_hat_tr)**2 +
            (Y_s_train - Ys_hat_tr)**2
        )),

        np.sqrt(mean_squared_error(Y_c_train, Yc_hat_tr)),
        np.sqrt(mean_squared_error(Y_r_train, Yr_hat_tr)),

        np.sqrt(np.mean(
            (Y_c_train - Yc_hat_tr)**2 +
            (Y_r_train - Yr_hat_tr)**2
        )),

        # Interval
        CRt,
        ERt,
        RIt,

        # r2
        r2_interval(Y_i_train, Yi_hat_tr),
        r2_interval(Y_s_train, Ys_hat_tr),

        (
            r2_interval(Y_i_train, Yi_hat_tr) +
            r2_interval(Y_s_train, Ys_hat_tr)
        ) / 2,

        r2_interval(Y_c_train, Yc_hat_tr),
        r2_interval(Y_r_train, Yr_hat_tr),

        (
            r2_interval(Y_c_train, Yc_hat_tr) +
            r2_interval(Y_r_train, Yr_hat_tr)
        ) / 2

    ])

    # =====================================================
    #  METRICS OOB
    # =====================================================

    oob_metrics = np.array([

        # RMSE
        np.sqrt(mean_squared_error(Y_i_test, Yi_hat)),
        np.sqrt(mean_squared_error(Y_s_test, Ys_hat)),

        np.sqrt(np.mean(
            (Y_i_test - Yi_hat)**2 +
            (Y_s_test - Ys_hat)**2
        )),

        np.sqrt(mean_squared_error(Y_c_test, Yc_hat)),
        np.sqrt(mean_squared_error(Y_r_test, Yr_hat)),

        np.sqrt(np.mean(
            (Y_c_test - Yc_hat)**2 +
            (Y_r_test - Yr_hat)**2
        )),

        # Interval
        CR,
        ER,
        RI,

        # r2
        r2_interval(Y_i_test, Yi_hat),
        r2_interval(Y_s_test, Ys_hat),

        (
            r2_interval(Y_i_test, Yi_hat) +
            r2_interval(Y_s_test, Ys_hat)
        ) / 2,

        r2_interval(Y_c_test, Yc_hat),
        r2_interval(Y_r_test, Yr_hat),

        (
            r2_interval(Y_c_test, Yc_hat) +
            r2_interval(Y_r_test, Yr_hat)
        ) / 2

    ])

################################################################
    # =====================================================
    #  NO-INFORMATION BASELINE
    # =====================================================

    # -----------------------------------------------------
    # LOWER E UPPER
    # -----------------------------------------------------

    baseline_L = np.mean(Y_i_train)

    baseline_U = np.mean(Y_s_train)

    # -----------------------------------------------------
    # CENTRE E RANGE
    # -----------------------------------------------------

    baseline_C = np.mean(Y_c_train)

    baseline_R = np.mean(Y_r_train)

    # -----------------------------------------------------
    # PREDICT BASELINE
    # -----------------------------------------------------

    Yi_baseline = np.full(
        len(Y_i_test),
        baseline_L
    )

    Ys_baseline = np.full(
        len(Y_s_test),
        baseline_U
    )

    Yc_baseline = np.full(
        len(Y_c_test),
        baseline_C
    )

    Yr_baseline = np.full(
        len(Y_r_test),
        baseline_R
    )

    # =====================================================
    #  METRICS NO-INFORMATION
    # =====================================================

    CR_base, ER_base, RI_base = interval_metrics(

        Y_i_test,
        Y_s_test,

        Yi_baseline,
        Ys_baseline
    )

    noinfo_metrics = np.array([

        # =================================================
        # RMSE
        # =================================================

        np.sqrt(mean_squared_error(
            Y_i_test,
            Yi_baseline
        )),

        np.sqrt(mean_squared_error(
            Y_s_test,
            Ys_baseline
        )),

        np.sqrt(np.mean(
            (Y_i_test - Yi_baseline)**2 +
            (Y_s_test - Ys_baseline)**2
        )),

        np.sqrt(mean_squared_error(
            Y_c_test,
            Yc_baseline
        )),

        np.sqrt(mean_squared_error(
            Y_r_test,
            Yr_baseline
        )),

        np.sqrt(np.mean(
            (Y_c_test - Yc_baseline)**2 +
            (Y_r_test - Yr_baseline)**2
        )),

        # =================================================
        # INTERVAL
        # =================================================

        CR_base,
        ER_base,
        RI_base,

        # =================================================
        # r2
        # =================================================

        r2_interval(Y_i_test, Yi_baseline),

        r2_interval(Y_s_test, Ys_baseline),

        (
            r2_interval(Y_i_test, Yi_baseline) +
            r2_interval(Y_s_test, Ys_baseline)
        ) / 2,

        r2_interval(Y_c_test, Yc_baseline),

        r2_interval(Y_r_test, Yr_baseline),

        (
            r2_interval(Y_c_test, Yc_baseline) +
            r2_interval(Y_r_test, Yr_baseline)
        ) / 2
    ])

################################################################

    # =====================================================
    #  0.632
    # =====================================================

    metrics632 = (
        0.368 * train_metrics +
        0.632 * oob_metrics
    )

    # =====================================================
    #  0.632+
    # =====================================================


    metrics632plus = []

    for i in range(len(train_metrics)):

        # =====================================================
        #  METRICS ERROR
        # =====================================================

        if metric_names[i].startswith("RMSE"):

            val = bootstrap_632_plus_error(

                train_metrics[i],
                oob_metrics[i],
                noinfo_metrics[i]
            )

        # =====================================================
        #  METRICS SCORE
        # =====================================================

        else:

            val = bootstrap_632_plus_score(

                train_metrics[i],
                oob_metrics[i],
                noinfo_metrics[i]
            )

        metrics632plus.append(val)

    metrics632plus = np.array(metrics632plus)

    # =====================================================
    #  SAVING
    # =====================================================

    all_train.append(train_metrics)

    all_oob.append(oob_metrics)

    all_632.append(metrics632)

    all_632plus.append(metrics632plus)

# =========================
#  MEAN (SD)
# =========================
print("\n" + "="*100)
print("RESULTS (AVG ± SD) - MONTE CARLO")
print("="*100)

all_train = np.array(all_train)

all_oob = np.array(all_oob)

all_632 = np.array(all_632)

all_632plus = np.array(all_632plus)

grupos = {

    "TRAIN": all_train,

    "BOOTSTRAP/OOB": all_oob,

    "BOOTSTRAP .632": all_632,

    "BOOTSTRAP .632+": all_632plus
}

for grupo, valores in grupos.items():

    print("\n" + "="*80)

    print(grupo)

    print("="*80)

    medias = np.mean(valores, axis=0)

    desvios = np.std(valores, axis=0)

    for i, nome in enumerate(metric_names):

        print(
            f"{nome}: "
            f"{medias[i]:.4f} ± {desvios[i]:.4f}"
        )